# CO₂–N₂O slab: how transmission changes with temperature (HITRAN · Voigt · Beer–Lambert)

Choose a **number density** (or pressure), a **path length**, the **CO₂ / N₂O amounts** and a **temperature range**; the notebook shows the transmittance of a CO₂ + N₂O gas slab as the temperature changes continuously, while the **incident spectrum stays fixed**. The wavelength axis is the same in every frame (**4.115–4.651 µm = 2150–2430 cm⁻¹**, both ν₃ bands).
**Run the cells top to bottom.**

> **Status, stated plainly.**
> * **Synthetic software tests** (equations, file parsing, interpolation, convergence, physical limits): run in §6; see the validation table printed there.
> * **Real HITRAN data and HAPI comparisons: NOT RUN by the author** (no HITRAN files, HAPI or internet were available). They are implemented in §9 and report **NOT RUN** until you supply data. No real-data number in this notebook has been verified.
> * The widgets were exercised with a stand-in for ipywidgets, not in a live Jupyter session.
> Everything labelled *setting* is an illustrative choice, not a measurement.

**Contents** 1 Atmospheric layer and ranges · 2 Theory · 3 Setup · 4 Physics engine · 5 HITRAN input/output · 6 Tests · 7 Load data and checks · 8 Interactive animation · 9 Checks on real data · 10 Export · 11 References and limitations

## 1. Which atmospheric layer, and which ranges? (the study)

**Layer: the troposphere.** Reasons, from the AFGL reference atmospheres (Anderson et al. 1986, AFGL-TR-86-0110, which tabulates six standard atmospheres):

1. **Uniform mixing ratios.** In those profiles CO₂ has a single constant mixing ratio (330 ppmv, a 1986 value), and N₂O stays close to its surface value
   (about 0.32 ppmv) through the troposphere and decreases with height in the stratosphere. A slab with *uniform* CO₂ and N₂O mole fractions
   is therefore a fair model of the troposphere, but not of the stratosphere.
2. **Conditions HITRAN is built for.** HITRAN's line parameters are referenced to 296 K and 1 atm; tropospheric temperatures and pressures bracket these.
3. **Simple line shapes.** At 0.09–1 bar the lines are Voigt profiles dominated by collisional (Lorentz) broadening; run `study_table()` in §7 after loading data to see
   the Lorentz/Doppler width ratio for your lines.

**Ranges sampled from the six AFGL atmospheres** (values read from a scanned copy and cross-checked against the ideal-gas density column of the report):

| quantity | tropospheric range | source values | slider (SI) |
|---|---|---|---|
| temperature | 194.8 – 299.7 K | coldest: tropical, 17 km geometric (93.7 hPa), tropical tropopause region; warmest: tropical, surface | 190 – 300 K, default sweep 195 – 300 K |
| pressure | 93.7 – 1018 hPa | tropical cold point … mid-latitude-winter surface (the six surface pressures are 1010–1018 hPa) | 9 000 – 102 000 Pa, default 54 000 Pa (≈ 5 km, U.S. Standard: 540.5 hPa); used in constant-pressure mode |
| number density | 3.49×10¹⁸ – 2.85×10¹⁹ cm⁻³ (3.5×10²⁴ – 2.85×10²⁵ m⁻³) | tropical cold point (17 km) … subarctic-winter surface; the report's density column | 3.2×10²⁴ – 3.2×10²⁵ m⁻³ (log), default 1.53×10²⁵ m⁻³ (U.S. Standard, 5 km); used in constant-density mode |
| path length | 1 cm – 10 km | laboratory cell … horizontal path in the layer | 0.01 – 10 000 m (logarithmic), default 100 m |

U.S. Standard (1976) reference points in the same tables (**geometric** altitude z): 0 km: 288.2 K, 1013 hPa · 5 km: 255.7 K, 540.5 hPa · 11 km: 216.8 K, 227.0 hPa.

**Altitude labels (corrected).** The ISA formulas use *geopotential* height h = r₀z/(r₀+z) (r₀ = 6356.766 km), the AFGL tables use *geometric* height z. They differ by 19 m at 11 km: 11 km geometric is 10.981 km geopotential (216.77 K), whereas the ISA tropopause (216.65 K, 22 632 Pa) is 11 km *geopotential* = 11.019 km *geometric*. `atmosphere_check()` (§7) recomputes the three U.S. Standard rows from the ISA formulas with this conversion (agreement 0.05 K, 0.02 % in the table; tested). The other four rows (tropical, mid-latitude winter, subarctic winter) could only be checked for internal consistency (n = P/k_BT matches the quoted density range), **not against the original report**. The coldest point of the temperature range (194.8 K, tropical 17 km) is the *tropical tropopause region*, not the middle of the troposphere; the ISA troposphere itself spans only 216.65–288.15 K.

**About the path length.** The Beer–Lambert slab needs T and P to be uniform along the path. That is natural for a *horizontal* path. A *vertical* path is not uniform:
in the U.S. Standard troposphere the temperature falls 6.5 K per km and the pressure drops by about 11 % in the first kilometre, so vertical paths longer than about 1 km need
a layered model (a later stage of this project). The range limits above are my choice, not a standard.

**Two ways to sweep the temperature (both are what-if experiments, not vertical profiles).**
* **Constant number density (default).** $P=nk_BT$ rises with $T$, as in a sealed, constant-volume cell (an *isochoric* process). The number of absorbers along the path, $nL$, is the same in every frame,
  so the changes come only from the line strengths, widths and shifts. In the AFGL atmospheres the surface pressures are all about the same while the surface temperatures differ by 42.5 K, so real air at a fixed pressure level changes
  density with temperature; the constant-density sweep isolates the *spectroscopic* effect of temperature.
* **Constant pressure.** An open cell: $n=P/(k_BT)$ rises on cooling, so the density effect adds to the spectroscopic effect.

The notebook prints the pressure range of the sweep and warns if it leaves the AFGL range (about 9 kPa to 102 kPa). The temperature range and the density (or pressure) are chosen independently, so an extreme combination is not necessarily a state found in the reference atmospheres.

**What the model still leaves out in the troposphere:** water vapour (up to roughly 0.8 % by volume near the surface in the U.S. Standard profile), clouds and aerosols, scattering, thermal emission of the gas.
Mole fractions: CO₂ 423.03 ppm (NOAA GML, Sep 2025) and N₂O 339.76 ppb (NOAA GML, Mar 2026), dry air, uniform.

**Spectral window (corrected).** The previous window, 4.300–4.430 µm = 2257–2326 cm⁻¹, started at the very top of the N₂O ν₃ band (centre ≈ 2224 cm⁻¹) and stopped inside the CO₂ ν₃ band (centre ≈ 2349 cm⁻¹), so it contained neither band. The window is now defined in wavenumber, **2150–2430 cm⁻¹ (4.115–4.651 µm)**:
* N₂O ν₃: centre 2224 cm⁻¹; P branch reaches below 2200, R branch about 2280 cm⁻¹ (rigid-rotor estimate with B ≈ 0.42 cm⁻¹).
* CO₂ ν₃: centre 2349 cm⁻¹; the R branch extends above 2400 cm⁻¹ (B ≈ 0.39 cm⁻¹: lines at 0.1 % of the strongest are near 2400 cm⁻¹ and still absorb on long paths), hence the upper edge 2430.
* Line-wing cut-off 20 cm⁻¹ → lines are needed from 2130 to 2450 cm⁻¹; the download range is 2125–2455 cm⁻¹ (`FETCH_MARGIN_CM1` = 25 ≥ `WING_CM1`; this is asserted in the setup cell, and `check_coverage()` compares it with `data_manifest.json`).

These band positions come from the standard band centres and rigid-rotor arithmetic; they are **confirmed against your own line list** by `check_band_edges()` and `window_content()` (§7, §9), not assumed.

## 2. Theory in brief

**Gas.** Ideal gas: $n = P/(k_BT)$, partial pressure $p_i = x_iP$, $n_i = x_i n$. At fixed $P$, cooling *raises* $n$.

**Line strength.** A line has position $\tilde\nu_0$ (cm$^{-1}$), lower-state energy $E''$ (cm$^{-1}$) and strength
$S$ in cm$^{-1}$/(molecule cm$^{-2}$) $=$ cm/molecule, with $\int\sigma\,d\tilde\nu=S$. HITRAN's $S$ already contains the natural isotopic
abundance, so $n$ is the *total* number density of the gas.

**Temperature scaling of $S$ (HITRAN definition):**
$$S(T)=S(T_{ref})\,\frac{Q(T_{ref})}{Q(T)}\;\frac{e^{-c_2E''/T}}{e^{-c_2E''/T_{ref}}}\;\frac{1-e^{-c_2\tilde\nu_0/T}}{1-e^{-c_2\tilde\nu_0/T_{ref}}},\qquad T_{ref}=296\,\mathrm K,\; c_2=\tfrac{hc}{k_B}=1.4388\ \mathrm{cm\,K}$$
* $Q(T)$ (partition function) grows with $T$ as more states are populated → lowers *every* line.
* $e^{-c_2E''/T}$ moves population to high-$E''$ levels → raises high-$E''$ lines, lowers low-$E''$ lines.
* The last factor is the stimulated-emission correction ($\approx1$ here since $c_2\tilde\nu_0/T\approx13$).
* There is **no universal trend**. With $Q\propto T$ a line's strength peaks at $T=c_2E''$. For an optically thin line, $W\propto nS$, so
  $d\ln W/d\ln T=-1+c_2E''/T$ at **constant number density** (lines with $E''>T/c_2$ strengthen on warming) and $-2+c_2E''/T$ at **constant pressure** (only lines with $E''>2T/c_2$ do). ($Q\propto T$ is only a rough approximation; real partition functions also contain vibrational terms.)

**Widths (always HWHM, never FWHM).**
$$\gamma_D=\frac{\tilde\nu_0}{c}\sqrt{\frac{2k_BT\ln2}{m}},\qquad
\gamma_L=\Big(\tfrac{T_{ref}}{T}\Big)^{n_{air}}\big[\gamma_{air}(P-p_{self})+\gamma_{self}\,p_{self}\big]\ (\text{pressures in atm}),\qquad \tilde\nu_0'=\tilde\nu_0+\delta_{air}P/\mathrm{atm}$$
The Gaussian (Doppler) part comes from the Maxwell velocity distribution; the Lorentzian part from collision-shortened state lifetimes. They are independent, so the line shape is their convolution, the Voigt profile
$$\varphi(\Delta\tilde\nu)=\frac{\mathrm{Re}\,w(z)}{\sigma_G\sqrt{2\pi}},\quad z=\frac{\Delta\tilde\nu+i\gamma_L}{\sigma_G\sqrt2},\quad \sigma_G=\frac{\gamma_D}{\sqrt{2\ln2}}$$
with $w$ the Faddeeva function. $\int\varphi\,d\tilde\nu=1$ exactly; a finite window misses a Lorentzian tail $\approx 2\gamma_L/(\pi W)$.

**Absorption and transmission.**
$$\sigma_i=\sum_k S_k(T)\varphi_k,\quad \alpha_i=n_i\sigma_i,\quad \alpha=\alpha_{CO_2}+\alpha_{N_2O},\quad \tau=\alpha L,\quad \mathcal T=e^{-\tau},\quad I=I_0\mathcal T$$
Intensity $I$ and transmittance $\mathcal T$ are different quantities; with a flat $I_0$ they have the same shape, with a blackbody $I_0$ they do not.
**Wavelength.** Everything is computed on HITRAN's wavenumber grid and plotted against $\lambda[\mu\mathrm m]=10^4/\tilde\nu[\mathrm{cm^{-1}}]$. A source spectrum per unit wavelength is
$B_\lambda=\frac{2hc^2}{\lambda^5}\big(e^{hc/\lambda k_BT}-1\big)^{-1}$; wavelength-weighted integrals use $d\lambda=10^4\,d\tilde\nu/\tilde\nu^2$.
Diagnostics: the transmittance at a selected wavelength, the wavelength-weighted window mean of $\mathcal T$, and the equivalent width $W=\int(1-\mathcal T)\,d\lambda$ (nm).
$W$ is **not** an absorbed energy.

**Thin vs saturated lines.** Thin: $W\approx SN$ ($N=nL$). Strongly saturated Lorentzian line: $W\approx2\sqrt{SN\gamma_L}$ (only wings still absorb).

**Numerical choices.** Each line is evaluated within ±`WING_CM1` of its centre. The profile value at the cut-off is subtracted (continuous cut-off), so that a line fades smoothly to zero and the spectrum has no tiny steps as pressure shifts move lines across grid points; the missing wing area is then about 2 × 2γ_L/(π·wing) of S. The threshold `S_MIN` is applied to S(T), not S(296 K). Grid step, wing and threshold are measured in §6 (synthetic lines) and §9 (real data).

**Temperature interpolation.** Spectra are computed line by line at *anchor* temperatures; between anchors a cubic spline of ln α(T) is used (α is smooth and roughly exponential in T). Error is measured by comparing with direct calculations half-way between anchors (the worst case).

**Where it fails.** Resolution effects at large $\tau$; CO₂ far wings (a truncated Voigt ignores sub-Lorentzian wings and line mixing);
hot gas (HITRAN lacks weak hot-band lines; use HITEMP); large mole fractions (HITRAN has no CO₂/N₂O cross-broadening, air is used instead);
no emission, scattering, water vapour, or continuum.

## 3. Setup

Install once (terminal or cell): `%pip install numpy scipy matplotlib ipywidgets hitran-api` — then restart the kernel. (No other package is needed. The live view uses `ipywidgets` only; ffmpeg/Pillow are needed only to save a movie file.)
If widgets do not render, see the fallback `draw()` at the start of §8.

In [5]:
import os, sys, json, glob, re, time, datetime, io, math, shutil, tempfile
from dataclasses import dataclass
from functools import lru_cache

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from scipy.interpolate import CubicSpline
from scipy.special import wofz

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML
    print("ipywidgets", widgets.__version__)
except ImportError:
    widgets = None
    print("ipywidgets missing: run  %pip install -U ipywidgets  and restart the kernel (the static draw() fallback still works)")
try:
    import hapi
    print("HAPI", getattr(hapi, "__version__", "?"))
except ImportError:
    hapi = None
    print("HAPI missing: run  %pip install hitran-api  (needed for partition sums and downloading)")
print("numpy", np.__version__)

ipywidgets 8.1.9
HAPI version: 1.3.0.0
To get the most up-to-date version please check http://hitran.org/hapi
ATTENTION: Python versions of partition sums from TIPS-2021 are now available in HAPI code

           MIT license: Copyright 2021 HITRAN team, see more at http://hitran.org. 

           If you use HAPI in your research or software development,
           please cite it using the following reference:
           R.V. Kochanov, I.E. Gordon, L.S. Rothman, P. Wcislo, C. Hill, J.S. Wilzewski,
           HITRAN Application Programming Interface (HAPI): A comprehensive approach
           to working with spectroscopic data, J. Quant. Spectrosc. Radiat. Transfer 177, 15-30 (2016)
           DOI: 10.1016/j.jqsrt.2016.03.005

           ATTENTION: This is the core version of the HITRAN Application Programming Interface.
                      For more efficient implementation of the absorption coefficient routine, 
                      as well as for new profiles, parameters and other f

In [6]:
# All illustrative settings (SI units unless stated). Edit here, then re-run from this cell downwards.
class cfg:
    # --- Spectral window: defined in WAVENUMBER (HITRAN's native unit); wavelength is derived ---
    # N2O nu3 band: centre ~2224 cm^-1 (P branch reaches below 2200, R branch to ~2280).
    # CO2 nu3 band: centre ~2349 cm^-1 (P branch to ~2290, R branch beyond 2400: the rigid-rotor estimate with
    # B = 0.39 cm^-1 puts lines at 0.1 % of the strongest line near 2400 cm^-1, and these still absorb on long paths).
    # Both bands, with their P and R branches, must fit inside the window: 2150-2430 cm^-1 = 4.115-4.651 um.
    # (Whether this really contains the bands is CHECKED against your line list: see check_band_edges().)
    NU_MIN = 2150.0           # cm^-1  (lambda = 4.651 um)
    NU_MAX = 2430.0           # cm^-1  (lambda = 4.115 um)
    LAM_MIN_UM = 1.0e4 / NU_MAX
    LAM_MAX_UM = 1.0e4 / NU_MIN
    NU_STEP = 0.0025          # cm^-1 grid step (checked by convergence_study(); Doppler HWHM is ~0.0015-0.0018 cm^-1)
    WING_CM1 = 20.0           # each line is evaluated within +/- this many cm^-1 (model choice; see convergence_study())
    S_MIN = 1.0e-26           # drop lines whose S(T) is below this [cm/molecule] (model choice, checked as well)
    SUBTRACT_EDGE = True      # continuous wing cut-off (see species_cross_section): no steps in the spectrum

    # --- Slider ranges (SI units) ---
    # 194.8 K is the AFGL *tropical tropopause region* (17 km), 299.7 K the tropical surface.
    # (The ISA/US-1976 tropopause is 216.65 K at 11 km geopotential height.)
    T_BOUNDS = (190.0, 300.0)      # K
    T_DEFAULT = (195.0, 300.0)     # K   : default animated sweep
    P_BOUNDS = (9000.0, 102000.0)  # Pa  : AFGL 93.7-1018 hPa, rounded outward
    P_DEFAULT = 54000.0            # Pa  : about 5 km in the U.S. Standard (540.5 hPa, AFGL Model 6)
    P_STEP = 500.0                 # Pa
    L_BOUNDS_LOG10_M = (-2.0, 4.0) # log10(L / m): 1 cm (laboratory cell) ... 10 km (horizontal path)
    L_DEFAULT_M = 100.0            # m
    # Sweep mode: "isochoric" = constant number density, P = n k_B T (sealed constant-volume cell);
    #             "isobaric"  = constant pressure (open cell). Both are what-if experiments, NOT atmospheric profiles.
    SWEEP_MODE = "isochoric"
    N_BOUNDS_LOG10_M3 = (24.5, 25.5)  # log10(n / m^-3)
    N_DEFAULT_M3 = 1.53e25            # m^-3: AFGL U.S. Standard at 5 km (1.532e19 cm^-3)

    # --- Animation ---
    N_PLAY = 211              # positions of the live Play button (0.5 K apart over the default 105 K range)
    PLAY_INTERVAL_MS = 60     # delay between Play steps (the real speed is limited by the rendering time)
    N_FRAMES = 96             # frames of the pre-rendered movie
    FPS = 24                  # 96 frames / 24 fps = exactly 4.0 s
    VIEW_BINS = 1400          # horizontal bins used for DRAWING (fixed bin edges: no jitter between frames)

    # --- Temperature interpolation ---
    # Only "anchor" spectra are computed line by line; spectra at other temperatures come from a cubic
    # spline of ln(alpha) versus T through the anchors (same physical model, see TemperatureInterpolator).
    ANCHOR_DT_MAX = 10.0      # K between exact spectra (error measured by interpolation_validation())
    ANCHOR_DLNP_MAX = 0.07    # max change of ln(P) between anchors (only matters if P varies)

    # --- Gas composition: dry-air mole fractions, assumed uniform in the slab ---
    X_CO2 = 423.03e-6         # NOAA GML global marine-surface monthly mean, Sep 2025 (value kept from the previous notebook)
    X_N2O = 339.76e-9         # NOAA GML global monthly mean, Mar 2026 (value kept from the previous notebook)
    # The remainder of the mixture is treated as non-absorbing air in this window.

    # --- Incident radiation (constant during the animation) ---
    T_SOURCE_K = 5700.0       # blackbody source temperature (user-chosen; close to the Sun's effective temperature)

    # --- Data ---
    DATA_DIR = "data"
    FETCH_MARGIN_CM1 = 25.0   # download this far beyond the window on each side (2125-2455 cm^-1); must be >= WING_CM1
    ALLOW_SYNTHETIC_DEMO = False   # True ONLY to practise with the widgets when no HITRAN files exist:
                                   # the spectra are then SYNTHETIC (invented lines), NOT HITRAN, NOT physical.

    SOURCES = {
        "profiles": "Anderson, G.P., Clough, S.A., Kneizys, F.X., Chetwynd, J.H., Shettle, E.P., "
                    "AFGL Atmospheric Constituent Profiles (0-120 km), AFGL-TR-86-0110, "
                    "Environmental Research Papers No. 954 (1986), https://apps.dtic.mil/sti/pdfs/ADA175173.pdf",
        "US_standard": "U.S. Standard Atmosphere, 1976 (U.S. Government Printing Office, Washington, DC)",
        "CO2": "NOAA GML, global monthly mean Sep 2025: 423.03 ppm, https://gml.noaa.gov/ccgg/trends/global.html",
        "N2O": "NOAA GML, global monthly mean Mar 2026: 339.76 ppb, https://www.gml.noaa.gov/ccgg/trends_n2o",
    }

# consistency guards (a silent violation would corrupt the results)
assert cfg.FETCH_MARGIN_CM1 >= cfg.WING_CM1, "download margin must be >= the line-wing cut-off, or the window edges lose wing contributions"
assert abs(cfg.FPS * 4 - cfg.N_FRAMES) == 0, "movie: N_FRAMES / FPS should be 4.0 s"
print("window %.0f-%.0f cm-1 = %.4f-%.4f um ; lines needed (window + wing): %.0f-%.0f cm-1 ; download: %.0f-%.0f cm-1"
      % (cfg.NU_MIN, cfg.NU_MAX, cfg.LAM_MIN_UM, cfg.LAM_MAX_UM, cfg.NU_MIN - cfg.WING_CM1, cfg.NU_MAX + cfg.WING_CM1,
         cfg.NU_MIN - cfg.FETCH_MARGIN_CM1, cfg.NU_MAX + cfg.FETCH_MARGIN_CM1))

window 2150-2430 cm-1 = 4.1152-4.6512 um ; lines needed (window + wing): 2130-2450 cm-1 ; download: 2125-2455 cm-1


## 4. Physics engine

Units: $\tilde\nu$ cm$^{-1}$; $S$ cm/molecule; $\varphi$ cm; $\sigma$ cm$^2$/molecule; $n$ cm$^{-3}$; $\alpha$ cm$^{-1}$; $L$ cm; widths are HWHM.
The second cell adds the **sweep model**: exact spectra at a few anchor temperatures + a spline of ln α, giving the spectrum at *any* temperature (this is what makes the animation continuous).

In [8]:
"""
rt_core.py - physics engine for the CO2 + N2O transmission model.

Everything here is plain NumPy/SciPy and does NOT need HITRAN data or HAPI,
so it can be unit-tested with synthetic lines (see tests/test_rt_core.py).

UNIT CONVENTIONS (HITRAN-style, CGS spectroscopy units)
  wavenumber nu ............ cm^-1
  line intensity S ......... cm^-1 / (molecule cm^-2)  == cm / molecule
  line-shape phi(nu) ....... cm            (integrates to 1 over cm^-1)
  cross section sigma ...... cm^2 / molecule   = S * phi
  number density n ......... molecule / cm^3
  absorption coeff alpha ... cm^-1         = n * sigma
  path length L ............ cm
  optical depth tau ........ dimensionless = alpha * L
  HWHM gamma ............... cm^-1  (HALF width at HALF maximum - NOT FWHM,
                                     NOT the 1/e half-width)
"""


# NumPy 2.x renamed trapz -> trapezoid; support both.
trapezoid = getattr(np, "trapezoid", None) or np.trapz

# ----------------------------------------------------------------------------
# Constants (SI values are exact since the 2019 SI redefinition)
# ----------------------------------------------------------------------------
H_PLANCK = 6.62607015e-34      # J s
C_LIGHT = 299792458.0          # m / s
K_B = 1.380649e-23             # J / K
AMU = 1.66053906660e-27        # kg (atomic mass unit; CODATA 2018 value)
ATM_PA = 101325.0              # Pa per standard atmosphere
T_REF = 296.0                  # K, HITRAN reference temperature
# second radiation constant c2 = h c / k_B, converted from m K to cm K
C2_CM_K = H_PLANCK * C_LIGHT / K_B * 100.0
LN2 = np.log(2.0)


# ----------------------------------------------------------------------------
# Data container for one gas
# ----------------------------------------------------------------------------
@dataclass
class LineList:
    """HITRAN line parameters for one molecule (all isotopologues).

    S is the HITRAN intensity at T_REF = 296 K. HITRAN's `sw` column already
    includes the natural isotopic abundance, so n_species below must be the
    TOTAL molecule number density (do not multiply by abundance again).
    """
    name: str
    M: int                      # HITRAN molecule id (CO2 = 2, N2O = 4)
    nu: np.ndarray              # cm^-1
    S: np.ndarray               # cm^-1/(molecule cm^-2) at 296 K
    E_low: np.ndarray           # cm^-1, lower-state energy
    gamma_air: np.ndarray       # cm^-1/atm, HWHM at 296 K, 1 atm
    gamma_self: np.ndarray      # cm^-1/atm, HWHM at 296 K, 1 atm
    n_air: np.ndarray           # temperature exponent of gamma_air
    delta_air: np.ndarray       # cm^-1/atm, pressure shift
    iso: np.ndarray             # local isotopologue index I (int)
    mass_amu: dict              # I -> molecular mass in amu

    def select(self, mask):
        return LineList(self.name, self.M, self.nu[mask], self.S[mask],
                        self.E_low[mask], self.gamma_air[mask],
                        self.gamma_self[mask], self.n_air[mask],
                        self.delta_air[mask], self.iso[mask], self.mass_amu)


# ----------------------------------------------------------------------------
# Ideal-gas mixture
# ----------------------------------------------------------------------------
def number_density_cm3(T, P_pa):
    """Total molecule number density n = P / (k_B T), returned in cm^-3.

    Assumes an ideal gas (valid here: ~1e5 Pa or less, 200-300 K, where the
    second virial correction is small). Partial pressure of species i is
    p_i = x_i P (Dalton) and n_i = x_i n for an ideal-gas mixture.
    """
    return P_pa / (K_B * T) * 1.0e-6        # m^-3 -> cm^-3


def dilution_warnings(x_co2, x_n2o, limit=0.01):
    """Air-broadening is used for the 'foreign' gas, which is only defensible
    if both absorbers are trace gases. Return warning strings otherwise."""
    if x_co2 < 0 or x_n2o < 0 or x_co2 + x_n2o > 1:
        raise ValueError("mole fractions must be >= 0 and sum to <= 1")
    msgs = []
    if x_co2 > limit or x_n2o > limit:
        msgs.append(
            "WARNING: x_CO2 or x_N2O exceeds %.0f%%. The model uses HITRAN "
            "air-broadening for all non-self collisions; CO2- and N2O-"
            "broadening of each other's lines is NOT available in the "
            "standard HITRAN line list, so results are an approximation "
            "and not validated in this regime." % (limit * 100))
    return msgs


# ----------------------------------------------------------------------------
# Temperature dependence of the line intensity (HITRAN definition)
#   S(T) = S(Tref) * [Q(Tref)/Q(T)]
#                  * exp(-c2 E''/T) / exp(-c2 E''/Tref)
#                  * [1 - exp(-c2 nu/T)] / [1 - exp(-c2 nu/Tref)]
# ----------------------------------------------------------------------------
def scale_intensity(S_ref, nu0, E_low, Q_ref_over_Q, T):
    """Scale HITRAN intensities from T_REF to T.

    Q_ref_over_Q : Q(T_REF)/Q(T), the partition-function ratio (per line)
    rB           : ratio of Boltzmann populations of the lower state
    rE           : ratio of stimulated-emission factors
    """
    rB = np.exp(-C2_CM_K * E_low * (1.0 / T - 1.0 / T_REF))
    rE = (-np.expm1(-C2_CM_K * nu0 / T)) / (-np.expm1(-C2_CM_K * nu0 / T_REF))
    return S_ref * Q_ref_over_Q * rB * rE


def partition_ratio_per_line(lines, T, partition_fn):
    """Q(T_REF)/Q(T) for every line (depends on the line's isotopologue).
    partition_fn(M, I, T) -> total internal partition sum (dimensionless)."""
    ratio = np.empty(len(lines.nu))
    for I in np.unique(lines.iso):
        r = partition_fn(lines.M, int(I), T_REF) / partition_fn(lines.M, int(I), T)
        ratio[lines.iso == I] = r
    return ratio


# ----------------------------------------------------------------------------
# Line widths
# ----------------------------------------------------------------------------
def doppler_hwhm(nu0, T, mass_amu):
    """Doppler HWHM [cm^-1] = (nu0/c) * sqrt(2 k_B T ln2 / m).
    (HWHM. The 1/e half-width is larger by 1/sqrt(ln2); FWHM is 2x HWHM.)"""
    m = np.asarray(mass_amu) * AMU
    return nu0 / C_LIGHT * np.sqrt(2.0 * K_B * T * LN2 / m)


def lorentz_hwhm(gamma_air, gamma_self, n_air, T, P_pa, x_self):
    """Collisional (Lorentz) HWHM [cm^-1] following the HITRAN definition:
         gamma_L = (Tref/T)^n_air * [ gamma_air (P - p_self) + gamma_self p_self ]
    with pressures in atm. 'Foreign' collisions are treated as air.
    The single exponent n_air is applied to the self part too (HITRAN
    provides no separate exponent in the standard list)."""
    P_atm = P_pa / ATM_PA
    p_self = x_self * P_atm
    return (T_REF / T) ** n_air * (gamma_air * (P_atm - p_self) + gamma_self * p_self)


# ----------------------------------------------------------------------------
# Voigt profile (Doppler Gaussian convolved with collisional Lorentzian)
# ----------------------------------------------------------------------------
def voigt_profile(dnu, gamma_D, gamma_L):
    """Normalised Voigt profile phi(dnu) [cm], with dnu = nu - nu0 [cm^-1].

    gamma_D, gamma_L are HWHM values. Gaussian sigma = gamma_D / sqrt(2 ln2).
    phi = Re[w(z)] / (sigma sqrt(2 pi)),   z = (dnu + i gamma_L) / (sigma sqrt 2)
    where w is the Faddeeva function (scipy.special.wofz).
    Integral of phi over all dnu is exactly 1.
    """
    sigma = gamma_D / np.sqrt(2.0 * LN2)
    z = (dnu + 1j * gamma_L) / (sigma * np.sqrt(2.0))
    return wofz(z).real / (sigma * np.sqrt(2.0 * np.pi))


# ----------------------------------------------------------------------------
# Cross section and absorption coefficient
# ----------------------------------------------------------------------------
def species_cross_section(grid, lines, T, P_pa, x_self, partition_fn,
                          wing_cm1=10.0, s_min=0.0, subtract_edge=False):
    """sigma(nu) [cm^2/molecule] = sum_i S_i(T) * phi_i(nu), natural abundance
    included via HITRAN's S. Each line is evaluated only within +/- wing_cm1
    of its (pressure-shifted) centre: this truncation is a MODEL CHOICE
    (it drops the far Lorentzian wings and ignores sub-Lorentzian behaviour).

    s_min is applied to the line strength AT THE GAS TEMPERATURE, S_i(T), not to the
    296 K value: a line with large E'' is weaker than its 296 K value when cold, and
    a threshold on S(296 K) would keep or drop the wrong lines.

    subtract_edge=True makes the cut-off CONTINUOUS: the profile value at the cut-off is subtracted,
    phi_cut(d) = phi(d) - phi(wing) for |d| < wing, so each line goes smoothly to zero at its edge.
    Without it a grid point that lies near a line's edge switches the line's far-wing contribution on or off
    when the pressure shift moves the line, which makes the spectrum a staircase function of T (tiny steps, but
    visible as flicker in weak wings at long path lengths). The price is a slightly larger missing wing area
    (about 2 x 2 gamma_L / (pi wing) of S instead of 2 gamma_L / (pi wing)); the cut-off is a model choice anyway.
    The default is False so that the original sum-rule test is unchanged; the dashboard uses cfg.SUBTRACT_EDGE.
    """
    P_atm = P_pa / ATM_PA
    in_range = (lines.nu > grid[0] - wing_cm1) & (lines.nu < grid[-1] + wing_cm1)
    L = lines.select(in_range)
    sigma = np.zeros_like(grid)
    if len(L.nu) == 0:
        return sigma
    S_T = scale_intensity(L.S, L.nu, L.E_low,
                          partition_ratio_per_line(L, T, partition_fn), T)
    strong = S_T >= s_min
    L, S_T = L.select(strong), S_T[strong]
    if len(L.nu) == 0:
        return sigma
    mass = np.array([L.mass_amu[int(i)] for i in L.iso])
    nu_c = L.nu + L.delta_air * P_atm             # HITRAN pressure shift
    gD = doppler_hwhm(nu_c, T, mass)
    gL = lorentz_hwhm(L.gamma_air, L.gamma_self, L.n_air, T, P_pa, x_self)
    lo = np.searchsorted(grid, nu_c - wing_cm1)
    hi = np.searchsorted(grid, nu_c + wing_cm1)
    edge = (voigt_profile(np.full(len(nu_c), float(wing_cm1)), gD, gL) if subtract_edge
            else np.zeros(len(nu_c)))
    for k in range(len(nu_c)):
        if hi[k] > lo[k]:
            sigma[lo[k]:hi[k]] += S_T[k] * (voigt_profile(
                grid[lo[k]:hi[k]] - nu_c[k], gD[k], gL[k]) - edge[k])
    return sigma


def absorption_coefficients(grid, co2_lines, n2o_lines, T, P_pa, x_co2, x_n2o,
                            partition_fn, wing_cm1=10.0, s_min=0.0, subtract_edge=False):
    """Return (alpha_CO2, alpha_N2O) in cm^-1.
    alpha_i = n_i * sigma_i,  n_i = x_i * P/(k_B T).
    The two are additive: alpha_total = alpha_CO2 + alpha_N2O."""
    n_tot = number_density_cm3(T, P_pa)
    a_co2 = x_co2 * n_tot * species_cross_section(
        grid, co2_lines, T, P_pa, x_co2, partition_fn, wing_cm1, s_min, subtract_edge)
    a_n2o = x_n2o * n_tot * species_cross_section(
        grid, n2o_lines, T, P_pa, x_n2o, partition_fn, wing_cm1, s_min, subtract_edge)
    return a_co2, a_n2o


# ----------------------------------------------------------------------------
# Beer-Lambert transmission
# ----------------------------------------------------------------------------
def beer_lambert(alpha_total, L_cm, I0):
    """tau = alpha L ; transmittance = exp(-tau) ; I = I0 * transmittance.
    Valid for a uniform, non-scattering, non-emitting slab and radiation
    that is narrow compared with the line structure (monochromatic)."""
    tau = alpha_total * L_cm
    trans = np.exp(-tau)
    return tau, trans, I0 * trans


def integrated_tau(grid, alpha, L_cm):
    """Integral of tau(nu) d nu  [cm^-1]  (a spectrally integrated optical depth).
    NOT an absorbed energy: that would need I0(nu) weighting and saturation."""
    return trapezoid(alpha * L_cm, grid)


def equivalent_width(grid, trans):
    """W = integral of (1 - T(nu)) d nu  [cm^-1]; spectral width of a fully
    opaque band that would absorb the same amount of a FLAT incident spectrum."""
    return trapezoid(1.0 - trans, grid)


def line_strength_vs_T(lines, k, Ts, partition_fn):
    """S(T) [cm/molecule] for line index k at each temperature in Ts."""
    one = lines.select(np.arange(len(lines.nu)) == k)
    out = []
    for T in Ts:
        qr = partition_ratio_per_line(one, T, partition_fn)
        out.append(float(scale_intensity(one.S, one.nu, one.E_low, qr, T)[0]))
    return np.array(out)


# ----------------------------------------------------------------------------
# Incident spectrum
# ----------------------------------------------------------------------------
def planck_per_cm1(nu_cm1, T_K):
    """Planck spectral radiance per unit wavenumber, W m^-2 sr^-1 (cm^-1)^-1.
    B = 2 h c^2 nu^3 / (exp(h c nu / k T) - 1) with nu in m^-1, times
    100 m^-1 per cm^-1 to convert the per-wavenumber unit."""
    nu_m = np.asarray(nu_cm1) * 100.0
    x = H_PLANCK * C_LIGHT * nu_m / (K_B * T_K)
    return 2.0 * H_PLANCK * C_LIGHT ** 2 * nu_m ** 3 / np.expm1(x) * 100.0


def planck_per_um(lam_um, T_K):
    """Planck spectral radiance per unit WAVELENGTH, W m^-2 sr^-1 um^-1:
    B = 2 h c^2 / lambda^5 / (exp(h c / (lambda k T)) - 1)   (lambda in m), times 1e-6 m/um."""
    lam = np.asarray(lam_um, dtype=float) * 1.0e-6
    x = H_PLANCK * C_LIGHT / (lam * K_B * T_K)
    return 2.0 * H_PLANCK * C_LIGHT ** 2 / lam ** 5 / np.expm1(x) * 1.0e-6


def make_incident_spectrum(grid_nu, flat, T_source):
    """I0 on the wavenumber grid, expressed per unit wavelength (plots are in wavelength).
    Returns (I0, unit string). flat=True gives I0 = 1 (arbitrary units)."""
    if flat:
        return np.ones_like(grid_nu), "arbitrary units (FLAT: I0 = 1)"
    return (planck_per_um(1.0e4 / grid_nu, T_source),
            "W m$^{-2}$ sr$^{-1}$ $\\mu$m$^{-1}$ (blackbody %.0f K)" % T_source)


def integrate_dlambda_um(grid_nu, y):
    """Integral of y over WAVELENGTH, in um, for y sampled on an increasing wavenumber grid:
    int y d(lambda) = int y * (1e4 / nu^2) d(nu)   (lambda[um] = 1e4 / nu[cm^-1])."""
    return trapezoid(np.asarray(y) * 1.0e4 / np.asarray(grid_nu) ** 2, grid_nu)


def window_mean_um(grid_nu, y):
    """Wavelength-weighted mean of y over the window (weights d(lambda))."""
    return integrate_dlambda_um(grid_nu, y) / integrate_dlambda_um(grid_nu, np.ones_like(grid_nu))


def pressure_at_constant_density(n_m3, Ts):
    """Pressure [Pa] of an ideal gas of FIXED number density n_m3 [molecules per m^3] at each temperature:
    P = n k_B T  (an isochoric process: a sealed, constant-volume cell). The absorber column n*L is then
    the same in every frame, so changes come from line strengths, widths and shifts only."""
    return float(n_m3) * K_B * np.asarray(Ts, dtype=float)


# ----------------------------------------------------------------------------
# Many animation frames from few exact calculations
# ----------------------------------------------------------------------------
def choose_anchors(Ts, Ps, dT_max=10.0, dlnP_max=0.07):
    """Indices of frames that get an EXACT spectrum calculation. Frames must be
    sorted by increasing T. Consecutive anchors are at most dT_max kelvin apart
    and at most dlnP_max apart in ln(P) (matters when P varies along the sweep).
    The first and last frame are always anchors."""
    Ts, Ps = np.asarray(Ts, float), np.asarray(Ps, float)
    n = len(Ts)
    if n < 3:
        return np.arange(n)
    idx = [0]
    for i in range(1, n):
        j = idx[-1]
        if abs(Ts[i] - Ts[j]) > dT_max or abs(np.log(Ps[i] / Ps[j])) > dlnP_max:
            idx.append(i - 1 if i - 1 > j else i)
    if idx[-1] != n - 1:
        idx.append(n - 1)
    return np.array(sorted(set(idx)))


class TemperatureInterpolator:
    """Spectrum at ANY temperature from a few exactly computed 'anchor' spectra.

    A cubic spline of ln(alpha) versus T is fitted through the anchors at every grid
    point (one spline, evaluated on demand). ln(alpha) is used because line strengths
    change roughly exponentially with T (Boltzmann factor); a spline of ln(alpha) follows that
    much better than a straight line in alpha and can never become negative.
    alpha_anchor[j] must be the exact spectrum at T_anchor[j] along the SAME path
    (P may change with T, e.g. constant density, as long as it is the same function of T).
    Values below rel_floor * max are treated as zero.

    This is an APPROXIMATION; its error is MEASURED by interpolation_validation().
    """
    def __init__(self, T_anchor, alpha_anchor, rel_floor=1e-12):
        self.T = np.asarray(T_anchor, float)
        A = np.asarray(alpha_anchor, float)
        self.A = A
        self.n_grid = A.shape[1]
        self.peak = A.max()
        self.floor = rel_floor * self.peak
        self.spline = None
        if self.peak > 0 and len(self.T) >= 2:
            self.spline = CubicSpline(self.T, np.log(np.maximum(A, self.floor)), axis=0)

    def __call__(self, T):
        T = np.asarray(T, float)
        if self.spline is None:
            return np.zeros((len(T), self.n_grid)) if T.ndim else np.zeros(self.n_grid)
        out = np.exp(self.spline(T))
        out[out < 2.0 * self.floor] = 0.0
        # at an anchor temperature return the exact anchor spectrum
        for j, Tj in enumerate(self.T):
            if T.ndim == 0:
                if abs(float(T) - Tj) < 1e-9:
                    out = self.A[j].copy()
            else:
                out[np.abs(T - Tj) < 1e-9] = self.A[j]
        return out


def interpolate_between_anchors(Ts, anchor_idx, anchor_values, rel_floor=1e-12):
    """Fill in the non-anchor frames by a cubic spline of ln(alpha) versus T
    (see TemperatureInterpolator). anchor_values: array (n_anchor, n_grid).
    Returns array (len(Ts), n_grid); anchor rows are returned exactly as computed."""
    Ts = np.asarray(Ts, float)
    A = np.asarray(anchor_values, float)
    anchor_idx = np.asarray(anchor_idx)
    if len(anchor_idx) == len(Ts):
        return A.copy()
    interp = TemperatureInterpolator(Ts[anchor_idx], A, rel_floor)
    out = interp(Ts)
    out[anchor_idx] = A
    return out


def envelope(x, y, n_bins=1500):
    """Reduce a long curve to 2*n_bins points (min and max of each bin) for
    plotting. Keeps every narrow peak and valley that a plain every-k-th-point
    thinning would miss. Output length is constant, so artists can be updated."""
    n = len(x)
    m = int(np.ceil(n / n_bins))
    if m <= 1:
        return np.asarray(x), np.asarray(y)
    pad = n_bins * m - n
    xp = np.pad(np.asarray(x, float), (0, pad), mode="edge").reshape(n_bins, m)
    yp = np.pad(np.asarray(y, float), (0, pad), mode="edge").reshape(n_bins, m)
    imin, imax = yp.argmin(axis=1), yp.argmax(axis=1)
    first, second = np.minimum(imin, imax), np.maximum(imin, imax)
    r = np.arange(n_bins)
    xs, ys = np.empty(2 * n_bins), np.empty(2 * n_bins)
    xs[0::2], ys[0::2] = xp[r, first], yp[r, first]
    xs[1::2], ys[1::2] = xp[r, second], yp[r, second]
    return xs, ys


def bin_stats(x, y, n_bins):
    """Min, max and mean of y in n_bins FIXED, equal-width bins (by index). Returns (x_centre, ymin, ymax, ymean).
    Unlike envelope(), the x positions do not depend on y, so successive animation frames line up
    exactly (no jitter). The mean of the transmittance is the spectrum seen at a coarser resolution."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    n = len(y)
    m = int(np.ceil(n / n_bins))
    nb = int(np.ceil(n / m))
    pad = nb * m - n
    xp = np.pad(x, (0, pad), mode="edge").reshape(nb, m)
    yp = np.pad(y, (0, pad), mode="edge").reshape(nb, m)
    return xp.mean(axis=1), yp.min(axis=1), yp.max(axis=1), yp.mean(axis=1)


# ----------------------------------------------------------------------------
# US Standard Atmosphere 1976, troposphere. NOTE: ISA formulas use GEOPOTENTIAL height h.
# The AFGL tables (and everyday altitude statements) use GEOMETRIC height z.
#   h = r0 z / (r0 + z),  r0 = 6356.766 km   (U.S. Standard Atmosphere 1976)
# The tropopause is at h = 11 km geopotential = 11.019 km geometric.
# ----------------------------------------------------------------------------
R0_US1976_M = 6356766.0


def geopotential_from_geometric_m(z_m):
    z = np.asarray(z_m, dtype=float)
    return R0_US1976_M * z / (R0_US1976_M + z)


def isa_troposphere(h_m):
    """Temperature [K] and pressure [Pa] in the ISA/US-1976 troposphere at GEOPOTENTIAL height h_m:
    T = 288.15 K - 0.0065 K/m * h ;  P = 101325 Pa * (T/288.15)^(g0 M0 / R* L)."""
    g0, M0, R_star, lapse = 9.80665, 0.0289644, 8.31432, 0.0065
    T = 288.15 - lapse * np.asarray(h_m, dtype=float)
    P = 101325.0 * (T / 288.15) ** (g0 * M0 / (R_star * lapse))
    return T, P


def isa_troposphere_geometric(z_m):
    """Same, for GEOMETRIC height z_m (what the AFGL tables use)."""
    return isa_troposphere(geopotential_from_geometric_m(z_m))

In [9]:
# Reference points of the AFGL atmospheres (Anderson et al. 1986, AFGL-TR-86-0110), read from a scanned copy by the previous author
# and NOT independently re-verified here (no access to the report). Altitudes are GEOMETRIC heights, as in the AFGL tables.
#   (label, z_geometric/km, T/K, P/hPa)
AFGL_REF = [("Tropical, surface",                          0, 299.7, 1013.0),
            ("Tropical, 17 km (tropical tropopause)",     17, 194.8,   93.7),
            ("Mid-latitude winter, surface",               0, 272.2, 1018.0),
            ("Subarctic winter, surface",                  0, 257.2, 1013.0),
            ("U.S. Standard, surface",                     0, 288.2, 1013.0),
            ("U.S. Standard, 5 km",                        5, 255.7,  540.5),
            ("U.S. Standard, 11 km (geometric)",          11, 216.8,  227.0)]
US_STD_ROWS = [4, 5, 6]          # indices of the U.S. Standard rows in AFGL_REF


def afgl_density_range_m3():
    """n = P / (k_B T) at every reference point [m^-3]: (min, max)."""
    n = [p * 100.0 / (K_B * T) for (_, _, T, p) in AFGL_REF]
    return min(n), max(n)


def atmosphere_check():
    """(1) number density at each AFGL point from the ideal-gas law; (2) for the U.S. Standard rows, the AFGL value
    against the ISA formulas evaluated at the same GEOMETRIC height (converted to geopotential height first).
    Row (2) is an internal consistency check of the notebook's tables, not an independent verification of the AFGL report."""
    print("%-40s %6s %8s %9s %12s | %9s %9s" % ("reference point (geometric height)", "z/km", "T/K", "P/Pa", "n/m^-3", "ISA dT/K", "ISA dP/%"))
    for k, (name, z, T, p) in enumerate(AFGL_REF):
        P = p * 100.0
        extra = ""
        if k in US_STD_ROWS:
            Ti, Pi = isa_troposphere_geometric(z * 1000.0)
            extra = "%9.2f %9.2f" % (T - float(Ti), 100.0 * (P - float(Pi)) / float(Pi))
        print("%-40s %6d %8.1f %9.0f %12.3e | %s" % (name, z, T, P, P / (K_B * T), extra))
    lo, hi = afgl_density_range_m3()
    print("density range of the table: %.3e - %.3e m^-3" % (lo, hi))
    h11 = float(geopotential_from_geometric_m(11000.0))
    print("11 km GEOMETRIC = %.3f km geopotential;  the ISA tropopause (216.65 K, 22632 Pa) is at 11 km GEOPOTENTIAL = %.3f km geometric"
          % (h11 / 1000.0, 1e-3 * R0_US1976_M * 11000.0 / (R0_US1976_M - 11000.0)))
    T1, P1 = isa_troposphere(1000.0)
    print("U.S. Standard 1 km up: dT = %.1f K, dP/P = %.1f %%  -> uniform-slab limit for vertical paths" % (288.15 - T1, 100 * (1 - P1 / 101325.0)))
    print("Scope of this model: a UNIFORM slab (one T, one P). The sweep modes are what-if experiments; e.g. 195 K at the density of 5 km "
          "does not occur in any reference atmosphere.")


def study_table(co2_lines=None, nu_ref=2300.0):
    """Number density, Doppler width and (if lines are loaded) a typical Lorentz width at each reference point."""
    gam = nq = None
    if co2_lines is not None:
        sel = (co2_lines.nu >= cfg.NU_MIN) & (co2_lines.nu <= cfg.NU_MAX)
        strong = sel & (co2_lines.S >= 1e-2 * co2_lines.S[sel].max())
        gam, nq = np.median(co2_lines.gamma_air[strong]), np.median(co2_lines.n_air[strong])
    print("%-40s %5s %7s %9s %12s %14s %14s %9s" % ("reference point", "z/km", "T/K", "P/Pa", "n/cm-3",
          "Doppler HWHM", "Lorentz HWHM", "ratio"))
    for name, z, T, p_hpa in AFGL_REF:
        P = p_hpa * 100.0
        gD = float(doppler_hwhm(nu_ref, T, 43.99))
        gL = None if gam is None else float(lorentz_hwhm(gam, gam, nq, T, P, 0.0))
        print("%-40s %5d %7.1f %9.0f %12.3e %14.2e %14s %9s" % (name, z, T, P, number_density_cm3(T, P), gD,
              "n/a (load data)" if gL is None else "%.2e" % gL, "" if gL is None else "%.1f" % (gL / gD)))
    print("Doppler HWHM for CO2 at %.0f cm-1. Lorentz HWHM: median gamma_air of the strong CO2 lines in the window." % nu_ref)

In [10]:
# The sweep model: exact spectra at a few "anchor" temperatures + a validated interpolator = a spectrum at ANY T
def build_conditions(P_pa, T_lo, T_hi, n_frames):
    """Fixed-pressure sweep: n_frames temperatures from T_lo to T_hi (ascending), constant P."""
    Ts = np.linspace(float(T_lo), float(T_hi), int(n_frames))
    return Ts, np.full(len(Ts), float(P_pa))


def build_conditions_isochoric(n_m3, T_lo, T_hi, n_frames):
    """Constant-number-density sweep: n_frames temperatures from T_lo to T_hi with P = n k_B T."""
    Ts = np.linspace(float(T_lo), float(T_hi), int(n_frames))
    return Ts, pressure_at_constant_density(n_m3, Ts)


def path_pressure(mode, n_m3, P_pa):
    """Return the function P(T) [Pa] describing the sweep."""
    if mode == "isochoric":
        return lambda T: pressure_at_constant_density(n_m3, T)
    return lambda T: np.full_like(np.asarray(T, float), float(P_pa))


def anchor_temperatures(T_lo, T_hi, P_of_T, dT_max, dlnP_max):
    """Anchor temperatures: both end points, neighbours at most dT_max kelvin and dlnP_max in ln(P) apart."""
    fine = np.linspace(T_lo, T_hi, max(3, int(round((T_hi - T_lo) / 0.5)) + 1))
    idx = choose_anchors(fine, P_of_T(fine), dT_max, dlnP_max)
    return fine[idx]


def default_lambda_sel_um(co2, n2o, grid):
    """Default selected wavelength [um]: centre of the strongest N2O line in the window
    (falls back to the strongest CO2 line, then to the window centre)."""
    for lines in (n2o, co2):
        inside = (lines.nu >= grid[0]) & (lines.nu <= grid[-1])
        if inside.any():
            return float(1e4 / lines.nu[inside][np.argmax(lines.S[inside])])
    return float(1e4 / grid[len(grid) // 2])


class SweepModel:
    """Absorption coefficients alpha_CO2(T), alpha_N2O(T) [cm^-1] for ANY T in [T_lo, T_hi].

    * At the anchor temperatures the spectra are computed line by line (the full physical model).
    * In between, TemperatureInterpolator (spline of ln alpha) is used. Its error is measured by
      interpolation_validation(); a frame is flagged `exact` only if it IS an anchor.
    * Path length L and the mole fractions are applied afterwards (they need no recomputation):
      alpha_i is proportional to x_i (exact except for the tiny x_i dependence of the self-broadening width,
      measured by test_mole_fraction_scaling_error), and tau = alpha L.
    * Pressure (or density), sweep mode and temperature range DO need a recomputation (line widths change)."""

    def __init__(self, grid, mode, n_m3, P_pa, T_lo, T_hi, x_co2_ref, x_n2o_ref, T_anchor, A_co2, A_n2o,
                 t_source=None, lam_sel_um=None, label="HITRAN"):
        self.grid, self.mode, self.n_m3, self.P_pa = grid, mode, float(n_m3), float(P_pa)
        self.T_lo, self.T_hi = float(T_lo), float(T_hi)
        self.x_co2_ref, self.x_n2o_ref = x_co2_ref, x_n2o_ref
        self.T_anchor = np.asarray(T_anchor, float)
        self.A_co2, self.A_n2o = np.asarray(A_co2), np.asarray(A_n2o)
        self.i_co2 = TemperatureInterpolator(self.T_anchor, self.A_co2)
        self.i_n2o = TemperatureInterpolator(self.T_anchor, self.A_n2o)
        self.t_source = cfg.T_SOURCE_K if t_source is None else t_source
        self.label = label
        self.lam_sel_um = lam_sel_um
        tot = self.A_co2 + self.A_n2o
        s = tot[:, ::20].ravel()
        s = s[s > 0]
        self.ylimD = (max(np.percentile(s, 1.0) / 10.0, 1e-14) if len(s) else 1e-14,
                      tot.max() * 3 if tot.max() > 0 else 1.0)
        self._P = path_pressure(mode, n_m3, P_pa)

    def P(self, T):
        return float(self._P(np.float64(T)))

    def temperatures(self, n):
        return np.linspace(self.T_lo, self.T_hi, int(n))

    def frame(self, T, x_co2=None, x_n2o=None):
        """dict(T, P, a_co2, a_n2o, exact) at temperature T (clipped to the computed range)."""
        T = float(np.clip(T, self.T_lo, self.T_hi))
        f1 = 1.0 if x_co2 is None else x_co2 / self.x_co2_ref
        f2 = 1.0 if x_n2o is None else x_n2o / self.x_n2o_ref
        return dict(T=T, P=self.P(T), a_co2=f1 * self.i_co2(T), a_n2o=f2 * self.i_n2o(T),
                    exact=bool(np.min(np.abs(self.T_anchor - T)) < 1e-9))


def build_sweep_model(grid, co2, n2o, mode, n_m3, P_pa, T_lo, T_hi, x_co2, x_n2o, partition_fn,
                      dT_max=None, dlnP_max=None, wing=None, s_min=None, verbose=True, label="HITRAN"):
    """Compute the anchor spectra exactly and wrap them in a SweepModel."""
    dT_max = cfg.ANCHOR_DT_MAX if dT_max is None else dT_max
    dlnP_max = cfg.ANCHOR_DLNP_MAX if dlnP_max is None else dlnP_max
    wing = cfg.WING_CM1 if wing is None else wing
    s_min = cfg.S_MIN if s_min is None else s_min
    if T_hi - T_lo < 1.0:
        T_hi = T_lo + 1.0                                   # a sweep needs a non-zero span
    # the reference mole fraction used for the computation must be > 0 so that later scaling is possible
    x_co2_ref = x_co2 if x_co2 > 0 else cfg.X_CO2
    x_n2o_ref = x_n2o if x_n2o > 0 else cfg.X_N2O
    P_of_T = path_pressure(mode, n_m3, P_pa)
    Ta = anchor_temperatures(T_lo, T_hi, P_of_T, dT_max, dlnP_max)
    t0 = time.time()
    A1, A2 = [], []
    for j, T in enumerate(Ta):
        P = float(P_of_T(T))
        a1, a2 = absorption_coefficients(grid, co2, n2o, T, P, x_co2_ref, x_n2o_ref, partition_fn,
                                         wing_cm1=wing, s_min=s_min, subtract_edge=cfg.SUBTRACT_EDGE)
        A1.append(a1)
        A2.append(a2)
        if verbose:
            print("  exact spectrum %2d/%d  T = %6.2f K  P = %8.1f Pa   (%.0f s elapsed)" % (j + 1, len(Ta), T, P, time.time() - t0))
    return SweepModel(grid, mode, n_m3, P_pa, T_lo, T_hi, x_co2_ref, x_n2o_ref, Ta, A1, A2,
                      lam_sel_um=default_lambda_sel_um(co2, n2o, grid), label=label)

## 5. HITRAN input/output and downloading

Lines are read from HITRAN's fixed-width 160-character format. HAPI's manual states that its default `.data` files use this format, so you can either download with HAPI **or** download manually from hitran.org (Data Access → Line-by-line, CO₂ and N₂O, all isotopologues, **2125–2455 cm⁻¹**, `.par`) and save as `data/CO2_2125_2455.par` and `data/N2O_2125_2455.par`. HAPI is always used for isotopologue masses and partition sums.

**Duplicates (corrected).** `load_linelist()` reads `<gas>*.data` *and* `<gas>*.par`. If a gas is present in both (or in two overlapping downloads), the old code summed every line twice. Now a record identical to one in an earlier file (all numbers and quantum numbers) is dropped; repeated records inside one file are kept (HITRAN lists physically distinct lines); the same transition with *different* numbers in two files (two database editions) raises an error. The count of removed duplicates is printed. Tested on synthetic files only.

This notebook contains **no API key**; see https://hitran.org/hapi/ if the download fails, and use the manual route as the fallback.

In [12]:
"""
py - read HITRAN line lists and talk to HAPI.

* Lines are read by our own parser from HITRAN's fixed-width 160-character
  .par format. HAPI's manual states that its default .data format is this same
  160-character format, so lines can come from HAPI's fetch OR from a manual
  download on hitran.org (.par files). A gas present in both is NOT double counted.
* HAPI is used only for the isotopologue table (masses, abundances) and for
  the partition sums Q(T) (TIPS, computed locally, no internet needed).
"""



# 0-based slice positions of the fields we need in a 160-char .par line
_SLICES = dict(M=(0, 2), I=(2, 3), nu=(3, 15), S=(15, 25), A=(25, 35),
               gamma_air=(35, 40), gamma_self=(40, 45), E_low=(45, 55),
               n_air=(55, 59), delta_air=(59, 67))


def _to_float(txt):
    """Float parser tolerant of Fortran 'D' exponents and HITRAN's odd '2.700-164'."""
    t = txt.strip().replace("D", "E")
    try:
        return float(t)
    except ValueError:
        m = re.fullmatch(r"(\d\.\d{3})-(\d{3})", t)
        if m:
            return float(m.group(1) + "E-" + m.group(2))
        raise


def _iso_index(ch):
    """Local isotopologue id: '1'-'9', '0' -> 10, 'A' -> 11, 'B' -> 12 ..."""
    if ch == "0":
        return 10
    if ch.isdigit():
        return int(ch)
    return 11 + ord(ch) - ord("A")


def _quanta_identity(line):
    """Quantum-number part of a 160-character record (global + local, columns 68-127), whitespace-normalised.
    Together with (molecule, isotopologue) it names a transition; used ONLY to detect the same transition
    delivered twice with different numbers (e.g. two HITRAN editions)."""
    return " ".join(line[67:127].split())


def read_par_file(path, M_expected):
    """Read one .par / .data file. Returns (records, n_unparsable).
    records = list of (strict_key, identity, row) where row is a dict of the parsed fields,
    strict_key = all numeric fields + quantum numbers (identical key == identical transition record)."""
    records, bad = [], 0
    with open(path, "r") as fh:
        for line in fh:
            s = line.rstrip("\r\n")
            if len(s) < 67:
                continue
            try:
                if int(s[0:2]) != M_expected:
                    continue
                row = {key: (_iso_index(s[a:b]) if key == "I" else int(s[a:b]) if key == "M" else _to_float(s[a:b]))
                       for key, (a, b) in _SLICES.items()}
            except (ValueError, IndexError):
                bad += 1
                continue
            q = _quanta_identity(s)
            numeric = tuple(row[k] for k in _SLICES)
            records.append(((numeric, q), (row["M"], row["I"], q) if q else None, row))
    return records, bad


def parse_par_files(paths, M_expected):
    """Parse .par / .data files into column arrays, WITHOUT double counting.

    Why: a gas can be present twice (HAPI's `<name>.data` and a manual `<name>.par`, or two overlapping
    downloads). Summing both would double every line in the overlap.
    Rules, in file order:
      * a record whose numbers AND quantum numbers equal a record of an EARLIER file is dropped
        (same transition delivered twice);
      * repeated records inside ONE file are kept (HITRAN lists physically distinct lines; nothing is removed
        within a file);
      * the same transition (molecule, isotopologue, quantum numbers) with DIFFERENT numbers in two files
        (different database editions) cannot be resolved automatically -> ValueError.
    Returns (cols, report)."""
    cols = {k: [] for k in _SLICES}
    seen_strict, seen_ident = set(), {}
    report = {"files": {}, "exact_duplicates_removed": 0, "unparsable_lines": 0}
    conflicts = []
    for path in paths:
        records, bad = read_par_file(path, M_expected)
        new_strict, new_ident, n_dup, n_conf, kept = set(), {}, 0, 0, 0
        for strict, ident, row in records:
            if strict in seen_strict:
                n_dup += 1
                continue
            if ident is not None and ident in seen_ident:
                n_conf += 1
                continue
            for k in _SLICES:
                cols[k].append(row[k])
            new_strict.add(strict)
            if ident is not None:
                new_ident.setdefault(ident, os.path.basename(path))
            kept += 1
        seen_strict |= new_strict
        for k, v in new_ident.items():
            seen_ident.setdefault(k, v)
        if n_conf:
            conflicts.append((os.path.basename(path), n_conf))
        report["files"][os.path.basename(path)] = {"records": len(records), "kept": kept, "exact_duplicates_removed": n_dup,
                                                   "conflicting_records": n_conf, "unparsable": bad}
        report["exact_duplicates_removed"] += n_dup
        report["unparsable_lines"] += bad
    if conflicts:
        raise ValueError("The same transitions appear with DIFFERENT parameters in several files (different database "
                         "editions?): %s. Keep only one file per gas in '%s'." % (conflicts, os.path.dirname(paths[0]) or "."))
    return {k: np.array(v) for k, v in cols.items()}, report


def repair_missing(cols):
    """Documented policy for unusable broadening parameters (<= 0 or NaN).

    HITRAN uses placeholder values where a parameter is not available. Rather
    than invent numbers, missing values are replaced by DATA-DERIVED estimates
    (median of valid lines of the same isotopologue, else of the same gas),
    and the number of replaced lines is returned so it is reported openly.
    gamma_self falls back to the (repaired) gamma_air; because x_self is
    ~1e-4 the effect on the width is of order x_self * |gamma_self - gamma_air|.
    """
    report = {}
    for key in ("gamma_air", "n_air"):
        bad = ~(cols[key] > 0)
        report[key] = int(bad.sum())
        for I in np.unique(cols["I"][bad]) if bad.any() else []:
            sel = cols["I"] == I
            good = sel & ~bad
            ref = (np.median(cols[key][good]) if good.any()
                   else np.median(cols[key][~bad]))
            cols[key][sel & bad] = ref
    bad = ~(cols["gamma_self"] > 0)
    report["gamma_self"] = int(bad.sum())
    cols["gamma_self"][bad] = cols["gamma_air"][bad]
    return report


def iso_table(M):
    """{local I: (global_id, name, abundance, mass_amu)} from HAPI's ISO table."""
    import hapi
    out = {}
    for (m, I), v in hapi.ISO.items():
        if m == M:
            out[I] = (v[hapi.ISO_INDEX["id"]], v[hapi.ISO_INDEX["iso_name"]],
                      v[hapi.ISO_INDEX["abundance"]], v[hapi.ISO_INDEX["mass"]])
    return out


def load_linelist(name, M, data_dir="data"):
    """Load all HITRAN files for one gas: <name>*.data and <name>*.par (never the .header files).
    Duplicates across files are removed (see parse_par_files). Returns (LineList, report, paths).
    report = {parameter: number of lines whose broadening parameter was replaced, ..., 'files': ...}."""
    paths = sorted(p for p in glob.glob(os.path.join(data_dir, name + "*.data"))
                   + glob.glob(os.path.join(data_dir, name + "*.par")))
    if not paths:
        raise FileNotFoundError(
            "No %s line list in '%s'. Run fetch_hitran(), or download a .par "
            "file from hitran.org and name it e.g. %s_2125_2425.par." % (name, data_dir, name))
    cols, dedup = parse_par_files(paths, M)
    if len(cols["nu"]) == 0:
        raise ValueError("Files %s contain no lines for molecule id %d" % (paths, M))
    report = repair_missing(cols)
    report["duplicates"] = dedup
    masses = {I: v[3] for I, v in iso_table(M).items()}
    order = np.argsort(cols["nu"], kind="stable")                  # sorted by wavenumber
    cols = {k: v[order] for k, v in cols.items()}
    lines = LineList(name, M, cols["nu"], cols["S"], cols["E_low"],
                     cols["gamma_air"], cols["gamma_self"], cols["n_air"],
                     cols["delta_air"], cols["I"].astype(int), masses)
    return lines, report, paths


def default_x(name):
    """Default mole fraction of the GAS named `name` (so CO2 never silently uses the N2O abundance)."""
    return {"CO2": cfg.X_CO2, "N2O": cfg.X_N2O}[name]


@lru_cache(maxsize=4096)
def _q_cached(M, I, T_key):
    import hapi as _h
    q = _h.partitionSum(M, I, T_key)
    if isinstance(q, tuple):
        q = q[1]
    return float(np.atleast_1d(q)[0])


def hapi_partition_fn(M, I, T):
    """Total internal partition sum Q(M, I, T) from HAPI's built-in TIPS (cached; T rounded to 1e-6 K).
    (Which TIPS edition is used depends on the installed HAPI version.)"""
    return _q_cached(int(M), int(I), round(float(T), 6))


def check_coverage(lines, nu_lo, nu_hi, wing, manifest_path=None):
    """Does the line list cover [nu_lo - wing, nu_hi + wing]?  Returns a dict.
    A line list cannot prove its own coverage (a region may simply have no lines), so the check
    uses the download range recorded in data_manifest.json when it exists; otherwise coverage is UNCONFIRMED."""
    need = (nu_lo - wing, nu_hi + wing)
    out = {"need": need, "first_line": float(lines.nu.min()), "last_line": float(lines.nu.max()),
           "largest_gap_cm-1": float(np.max(np.diff(np.sort(lines.nu)))) if len(lines.nu) > 1 else float("nan"),
           "confirmed": None, "downloaded": None}
    if manifest_path and os.path.exists(manifest_path):
        with open(manifest_path) as fh:
            rng = json.load(fh).get("wavenumber_range_cm-1")
        if rng:
            out["downloaded"] = tuple(rng)
            out["confirmed"] = bool(rng[0] <= need[0] and rng[1] >= need[1])
    return out


def check_band_edges(lines, nu_lo, nu_hi, rel=1e-3, edge=5.0, T=296.0):
    """Is a band cut off by the window?  Counts lines with S >= rel * (strongest S in the window) that lie within
    `edge` cm^-1 of either window edge (inside or outside). If there are none, the strong part of the band
    ends well inside the window. Returns a dict (uses 296 K strengths)."""
    inw = (lines.nu >= nu_lo) & (lines.nu <= nu_hi)
    if not inw.any():
        return {"lines_in_window": 0, "strong_lines_near_edges": None, "S_max": 0.0}
    smax = lines.S[inw].max()
    strong = lines.S >= rel * smax
    near_lo = strong & (np.abs(lines.nu - nu_lo) <= edge)
    near_hi = strong & (np.abs(lines.nu - nu_hi) <= edge)
    return {"lines_in_window": int(inw.sum()), "S_max": float(smax), "strong_lines_near_low_edge": int(near_lo.sum()),
            "strong_lines_near_high_edge": int(near_hi.sum()),
            "strong_lines_near_edges": int(near_lo.sum() + near_hi.sum())}

In [13]:
MOLECULES = (("CO2", 2), ("N2O", 4))      # HITRAN molecule ids

def fetch_hitran():
    # Download CO2 and N2O line lists with HAPI and write data/data_manifest.json.
    try:
        import hapi
    except ImportError:
        raise RuntimeError("HAPI not installed. Run: %pip install hitran-api")
    os.makedirs(cfg.DATA_DIR, exist_ok=True)
    hapi.db_begin(cfg.DATA_DIR)
    stale = glob.glob(os.path.join(cfg.DATA_DIR, "CO2*.par")) + glob.glob(os.path.join(cfg.DATA_DIR, "N2O*.par"))
    if stale:
        print("Note: manual .par files exist (%s); duplicates between them and the HAPI .data files are removed automatically when loading." % stale)
    numin, numax = cfg.NU_MIN - cfg.FETCH_MARGIN_CM1, cfg.NU_MAX + cfg.FETCH_MARGIN_CM1
    hapi.VARIABLES["DISPLAY_FETCH_URL"] = True
    manifest = {"retrieval_date": datetime.date.today().isoformat(), "source": "HITRANonline via HAPI",
                "hapi_version": getattr(hapi, "__version__", "unknown"),
                "wavenumber_range_cm-1": [numin, numax],
                "units": {"nu": "cm-1", "S": "cm-1/(molecule cm-2) at 296 K, abundance-weighted",
                          "gamma": "cm-1/atm HWHM at 296 K", "E_low": "cm-1"}, "gases": {}}
    for name, M in MOLECULES:
        iso = iso_table(M)
        table = "%s_%d_%d" % (name, round(numin), round(numax))
        try:
            hapi.fetch_by_ids(table, [v[0] for v in iso.values()], numin, numax)
        except Exception as exc:
            raise RuntimeError("Download failed for %s: %s\nUse the manual .par route described above." % (name, exc))
        manifest["gases"][name] = {"hitran_molecule_id": M, "table": table,
            "isotopologues": {str(I): {"global_id": v[0], "name": v[1], "abundance": v[2], "mass_amu": v[3]}
                              for I, v in iso.items()}}
    for name, M in MOLECULES:
        lines, report, paths = load_linelist(name, M, cfg.DATA_DIR)
        manifest["gases"][name].update(n_lines=int(len(lines.nu)), files=[os.path.basename(p) for p in paths],
                                       parameters_replaced_by_data_derived_estimate=report,
                                       exact_duplicates_removed=dup["exact_duplicates_removed"])
        dup = report.pop("duplicates")
        print("%s: %d lines; exact duplicates removed: %d; replaced parameters: %s" % (name, len(lines.nu), dup["exact_duplicates_removed"], report))
    with open(os.path.join(cfg.DATA_DIR, "data_manifest.json"), "w") as fh:
        json.dump(manifest, fh, indent=2)
    print("Wrote", os.path.join(cfg.DATA_DIR, "data_manifest.json"))

## 6. Tests

All tests in this section use **synthetic** lines and a toy partition function (`toy_Q`); they test the equations and the code, **not HITRAN and not the real spectrum**. The 31 original tests are unchanged; 31 new tests cover line-list integrity, spectral coverage, atmosphere table, interpolation accuracy, grid/wing/threshold convergence, continuity, and physical limits. Tests named `test_conv_*` are *numerical convergence* tests (shown as a separate category); they also use invented lines.

In [15]:
# ===== SYNTHETIC test fixtures. Nothing in this cell is HITRAN data. =====
# The line lists below are INVENTED rigid-rotor-like bands with plausible orders of magnitude. They exist so that the
# code (equations, interpolation, convergence, file parsing) can be tested without data. Never use them as spectra.
def toy_Q(M, I, T):
    return T


def synthetic_band(M, iso, mass, nu0, B, S_peak, even_only, Jmax, E_extra=0.0, S_scale=1.0):
    """P and R branch of a rigid-rotor-like band (lower-state energy E'' = B J(J+1) + E_extra)."""
    rows = []
    for J in range(0, Jmax + 1):
        if even_only and J % 2:
            continue
        E = B * J * (J + 1) + E_extra
        for branch in ("R", "P"):
            if branch == "P" and J == 0:
                continue
            nu = nu0 + 2 * B * (J + 1) if branch == "R" else nu0 - 2 * B * J
            hl = (J + 1) if branch == "R" else J                     # Honl-London-like factor
            S = hl * np.exp(-C2_CM_K * B * J * (J + 1) / T_REF)      # Boltzmann factor of the lower rotational level
            gam = max(0.04, 0.078 - 0.0006 * min(J, 60))
            rows.append((nu, S, E, gam, 1.2 * gam, 0.70, -0.003, iso))
    r = np.array(rows)
    r[:, 1] *= S_peak * S_scale / r[:, 1].max()
    return r


def synthetic_linelists():
    """(CO2_like, N2O_like) LineLists. SYNTHETIC. CO2-like: main band near 2349 cm^-1, a weaker 'isotopologue' band near
    2283 and a 'hot band' with large E''. N2O-like: main band near 2224 cm^-1 plus a hot-band-like copy."""
    def pack(name, M, parts, masses):
        r = np.vstack(parts)
        r = r[np.argsort(r[:, 0])]
        return LineList(name, M, r[:, 0], r[:, 1], r[:, 2], r[:, 3], r[:, 4], r[:, 5], r[:, 6], r[:, 7].astype(int), masses)
    co2 = pack("CO2", 2, [synthetic_band(2, 1, 44.0, 2349.14, 0.3902, 3.5e-18, True, 80),
                          synthetic_band(2, 2, 45.0, 2283.49, 0.3790, 3.5e-18, True, 80, S_scale=0.0111),
                          synthetic_band(2, 1, 44.0, 2336.0, 0.3910, 3.5e-18, True, 70, E_extra=667.0, S_scale=0.07)],
               {1: 44.0, 2: 45.0})
    n2o = pack("N2O", 4, [synthetic_band(4, 1, 44.0, 2223.76, 0.4190, 1.5e-18, False, 90),
                          synthetic_band(4, 1, 44.0, 2209.0, 0.4180, 1.5e-18, False, 70, E_extra=589.0, S_scale=0.08)],
               {1: 44.0, 2: 45.0})
    return co2, n2o


def format_par_line(M, I, nu, S, A, g_air, g_self, E, n_air, delta, quanta="", iref="0000000000"):
    """Build one 160-character HITRAN-style record (used to test the parser and the duplicate handling)."""
    g5 = lambda g: ("%.4f" % g).lstrip("0").rjust(5)          # HITRAN writes 0.0750 as ".0750" (5 characters)
    head = "%2d%1d%12.6f%10.3E%10.3E%s%s%10.4f%4.2f%8.5f" % (M, I, nu, S, A, g5(g_air), g5(g_self), E, n_air, delta)
    body = ("%-60s" % quanta) + ("%6d" % 0) + ("%12s" % iref)
    return (head + " " * (67 - len(head)) + body).ljust(160)[:160]


def write_par(path, rows, M=2):
    """rows: iterable of (I, nu, S, E, quanta). Other parameters are fixed placeholders (SYNTHETIC)."""
    with open(path, "w") as fh:
        for (I, nu, S, E, q) in rows:
            fh.write(format_par_line(M, I, nu, S, 1.0, 0.07, 0.09, E, 0.7, -0.003, q) + "\n")
print("synthetic fixtures defined (test use only):", [len(x.nu) for x in synthetic_linelists()], "lines (CO2-like, N2O-like)")

synthetic fixtures defined (test use only): [233, 322] lines (CO2-like, N2O-like)


In [16]:
# ===== Validation registry =====
CAT_SYN = "Synthetic software test"
CAT_CONV = "Numerical convergence (synthetic lines)"
CAT_REAL = "Real HITRAN data"
CAT_HAPI = "HAPI comparison"
VALIDATION = []          # (category, name, status, metric)

def record(category, name, status, metric=""):
    VALIDATION[:] = [r for r in VALIDATION if not (r[0] == category and r[1] == name)]
    VALIDATION.append((category, name, status, metric))

def validation_table(categories=None):
    rows = [r for r in VALIDATION if categories is None or r[0] in categories]
    print("%-40s %-52s %-8s %s" % ("category", "check", "status", "metric"))
    print("-" * 150)
    for c, n, s, m in sorted(rows, key=lambda r: (r[0], r[1])):
        print("%-40s %-52s %-8s %s" % (c, n, s, m))
    print("\nPASS %d | FAIL %d | NOT RUN %d" % tuple(sum(1 for r in rows if r[2] == k) for k in ("PASS", "FAIL", "NOT RUN")))


# ===== Numerical studies: work on ANY line lists (synthetic now, real HITRAN later) =====
def _trans_metrics(g, a_ref, a_var, k, L_cm_list):
    """Compare a variant spectrum (grid g) with the reference sampled at the same wavenumbers (every k-th point)."""
    a_ref = a_ref[::k][:len(g)]
    out = {}
    core = a_ref > 1e-3 * a_ref.max()
    out["alpha_rel_core"] = float(np.max(np.abs(a_var - a_ref)[core] / a_ref[core])) if core.any() else 0.0
    for L in L_cm_list:
        t_ref, t_var = np.exp(-a_ref * L), np.exp(-a_var * L)
        ew_ref = trapezoid(1.0 - t_ref, g)
        out[L] = (float(np.max(np.abs(t_var - t_ref))),                                   # max |dT| (pointwise)
                  float(abs(trapezoid(1.0 - t_var, g) - ew_ref) / max(ew_ref, 1e-30)),    # relative error of the equivalent width
                  float(abs(window_mean_um(g, t_var) - window_mean_um(g, t_ref))))        # error of the window-mean transmittance
    return out


def convergence_study(co2, n2o, partition_fn, window, conditions, L_m_list=(1.0, 100.0, 1.0e4),
                      x_co2=None, x_n2o=None, step=None, wing=None, s_min=None, verbose=True):
    """Error of the three numerical choices (grid step, line-wing cut-off, weak-line threshold) inside `window` (cm^-1).

    REFERENCE = step/4, wing x4, s_min/1000.  Each row changes ONE choice from the baseline (cfg values); the
    'baseline' row therefore shows the TOTAL numerical error of the production settings.
    conditions: list of (T [K], P [Pa]).  Reported for every path length: max |dTransmittance| over the window,
    relative error of the equivalent width, error of the window-mean transmittance.
    Returns dict {(variant, T, L): (max dT, rel EW error, |d mean T|)} and prints a table."""
    step = cfg.NU_STEP if step is None else step
    wing = cfg.WING_CM1 if wing is None else wing
    s_min = cfg.S_MIN if s_min is None else s_min
    x1 = default_x("CO2") if x_co2 is None else x_co2
    x2 = default_x("N2O") if x_n2o is None else x_n2o
    lo, hi = window
    variants = ([("step x4", dict(step=step * 4)), ("step x2", dict(step=step * 2)), ("baseline", {}), ("step /2", dict(step=step / 2))]
                + [("wing /4", dict(wing=wing / 4)), ("wing /2", dict(wing=wing / 2)), ("wing x2", dict(wing=wing * 2))]
                + [("S_min x1000", dict(s=s_min * 1e3)), ("S_min x100", dict(s=s_min * 1e2)), ("S_min x10", dict(s=s_min * 10)),
                   ("S_min /10", dict(s=s_min / 10))])
    res = {}
    if verbose:
        print("window %.1f-%.1f cm-1 | baseline step %.5g cm-1, wing %.3g cm-1, S_min %.1e | reference: step/4, wing x4, S_min/1000"
              % (lo, hi, step, wing, s_min))
        print("%-12s %7s %9s %8s %12s %12s %12s %12s" % ("variant", "T/K", "P/Pa", "L/m", "max|dTrans|", "rel dEW", "|d meanT|", "rel d alpha"))
    for (T, P) in conditions:
        g_ref = np.arange(lo, hi, step / 4)
        a1, a2 = absorption_coefficients(g_ref, co2, n2o, T, P, x1, x2, partition_fn, wing_cm1=wing * 4, s_min=s_min / 1e3, subtract_edge=cfg.SUBTRACT_EDGE)
        a_ref = a1 + a2
        for name, ch in variants:
            st, wg, sm = ch.get("step", step), ch.get("wing", wing), ch.get("s", s_min)
            k = int(round(st / (step / 4)))
            g = np.arange(lo, hi, st)
            b1, b2 = absorption_coefficients(g, co2, n2o, T, P, x1, x2, partition_fn, wing_cm1=wg, s_min=sm, subtract_edge=cfg.SUBTRACT_EDGE)
            m = _trans_metrics(g, a_ref, b1 + b2, k, [Lm * 100.0 for Lm in L_m_list])
            for Lm in L_m_list:
                res[(name, T, Lm)] = m[Lm * 100.0] + (m["alpha_rel_core"],)
                if verbose:
                    e = m[Lm * 100.0]
                    print("%-12s %7.1f %9.0f %8.3g %12.3e %12.3e %12.3e %12.3e" % (name, T, P, Lm, e[0], e[1], e[2], m["alpha_rel_core"]))
    return res


def interpolation_validation(co2, n2o, partition_fn, grid, mode, n_m3, P_pa, T_lo, T_hi, x_co2, x_n2o,
                             dT_list=(20.0, 10.0, 5.0), L_m_list=(1.0, 100.0, 1.0e4), verbose=True):
    """Error of the temperature interpolation, measured where it is WORST: half way between two anchors.
    For each anchor spacing dT the model is built, then the spectrum at every mid-point temperature is computed
    directly (full line-by-line) and compared with the interpolated one.
    Returns {dT: (max |dTrans|, max relative error of alpha in the core)} over all mid-points and path lengths."""
    out = {}
    if verbose:
        print("%-8s %9s %12s %14s" % ("dT/K", "anchors", "max|dTrans|", "max rel dalpha"))
    for dT in dT_list:
        mod = build_sweep_model(grid, co2, n2o, mode, n_m3, P_pa, T_lo, T_hi, x_co2, x_n2o, partition_fn,
                                dT_max=dT, dlnP_max=10.0, verbose=False)
        worst_t, worst_a = 0.0, 0.0
        for Tm in 0.5 * (mod.T_anchor[:-1] + mod.T_anchor[1:]):
            P = mod.P(Tm)
            e1, e2 = absorption_coefficients(grid, co2, n2o, Tm, P, mod.x_co2_ref, mod.x_n2o_ref, partition_fn,
                                             wing_cm1=cfg.WING_CM1, s_min=cfg.S_MIN, subtract_edge=cfg.SUBTRACT_EDGE)
            ex = e1 + e2
            fr = mod.frame(Tm)
            ip = fr["a_co2"] + fr["a_n2o"]
            core = ex > 1e-3 * ex.max()
            worst_a = max(worst_a, float(np.max(np.abs(ip - ex)[core] / ex[core])))
            for Lm in L_m_list:
                worst_t = max(worst_t, float(np.max(np.abs(np.exp(-ip * Lm * 100.0) - np.exp(-ex * Lm * 100.0)))))
        out[dT] = (worst_t, worst_a)
        if verbose:
            print("%-8.1f %9d %12.3e %14.3e" % (dT, len(mod.T_anchor), worst_t, worst_a))
    return out


def mole_fraction_scaling_error(co2, n2o, partition_fn, grid, T, P_pa, x_ref, x_new, gas="CO2", L_m=100.0):
    """The dashboard rescales alpha_i linearly with x_i. Only the self-broadening width depends on x_i, so the
    rescaling is not exact. Returns (max rel. error of alpha in the core, max |dTransmittance| at L_m)."""
    lines = co2 if gas == "CO2" else n2o
    ref = x_ref * number_density_cm3(T, P_pa) * species_cross_section(grid, lines, T, P_pa, x_ref, partition_fn, cfg.WING_CM1, cfg.S_MIN, cfg.SUBTRACT_EDGE)
    new = x_new * number_density_cm3(T, P_pa) * species_cross_section(grid, lines, T, P_pa, x_new, partition_fn, cfg.WING_CM1, cfg.S_MIN, cfg.SUBTRACT_EDGE)
    scaled = ref * (x_new / x_ref)
    core = new > 1e-3 * new.max()
    return (float(np.max(np.abs(scaled - new)[core] / new[core])),
            float(np.max(np.abs(np.exp(-scaled * L_m * 100.0) - np.exp(-new * L_m * 100.0)))))

In [17]:
def toy_Q(M, I, T):
    return T


def make_lines(nu, S, E=0.0, g_air=0.07, g_self=0.07, n=0.7, delta=0.0, M=2, mass=44.0):
    nu = np.atleast_1d(np.asarray(nu, float))
    k = len(nu)
    full = lambda v: np.full(k, v, float) if np.isscalar(v) else np.asarray(v, float)
    return LineList("toy", M, nu, full(S), full(E), full(g_air), full(g_self),
                       full(n), full(delta), np.ones(k, int), {1: mass})


# ---------------------------------------------------------------- line shape
def _norm_with_tail(gD, gL, W, step):
    x = np.arange(-W, W + step, step)
    area = trapezoid(voigt_profile(x, gD, gL), x)
    return area, area + 2 * gL / (np.pi * W)       # + analytic Lorentzian tail


def test_voigt_normalisation():
    # Raw integral over +/-W misses the Lorentzian tail ~ 2 gL/(pi W);
    # adding that tail analytically must give 1 to high accuracy.
    for gD, gL in [(0.002, 0.04), (0.01, 0.001), (0.01, 0.0), (0.002, 0.2)]:
        raw, corrected = _norm_with_tail(gD, gL, W=2000.0, step=min(gD, max(gL, 1e-3)) / 8)
        assert abs(corrected - 1.0) < 2e-5, (gD, gL, raw, corrected)
        assert raw <= 1.0 + 1e-9


def test_voigt_limits_and_hwhm_convention():
    # pure Gaussian: peak 1/(sigma sqrt(2 pi)); value at dnu = gD must be half the peak (HWHM!)
    gD = 0.01
    peak = voigt_profile(0.0, gD, 0.0)
    assert abs(voigt_profile(gD, gD, 0.0) / peak - 0.5) < 1e-12
    sigma = gD / np.sqrt(2 * np.log(2))
    assert abs(peak - 1 / (sigma * np.sqrt(2 * np.pi))) < 1e-9 * peak
    # pure Lorentzian: peak 1/(pi gL); half maximum at dnu = gL
    gL = 0.05
    p = voigt_profile(0.0, 1e-7, gL)
    assert abs(p - 1 / (np.pi * gL)) / p < 1e-4
    assert abs(voigt_profile(gL, 1e-7, gL) / p - 0.5) < 1e-4


def test_doppler_width_scaling_and_value():
    a = doppler_hwhm(2349.0, 250.0, 44.0)
    b = doppler_hwhm(2349.0, 1000.0, 44.0)
    assert abs(b / a - 2.0) < 1e-12                       # proportional to sqrt(T)
    ref = 2349.0 / C_LIGHT * np.sqrt(2 * 1.380649e-23 * 250 * np.log(2) / (44.0 * 1.66053906660e-27))
    assert abs(a - ref) / ref < 1e-12


# ------------------------------------------------------- intensity scaling
def test_intensity_scaling_identity_and_formula():
    S = scale_intensity(1e-20, 2300.0, 500.0, 1.0, T_REF)
    assert abs(S - 1e-20) < 1e-30                          # T = T_ref -> unchanged
    T, nu, E, S0 = 230.0, 2224.0, 300.0, 2e-19
    c2 = 1.438776877
    expected = (S0 * (T_REF / T) * np.exp(-c2 * E / T) / np.exp(-c2 * E / T_REF)
                * (1 - np.exp(-c2 * nu / T)) / (1 - np.exp(-c2 * nu / T_REF)))
    got = scale_intensity(S0, nu, E, toy_Q(0, 0, T_REF) / toy_Q(0, 0, T), T)
    assert abs(got - expected) / expected < 1e-8


def test_c2_value():
    assert abs(C2_CM_K - 1.438776877) < 1e-8


def test_temperature_trends_differ_between_transitions():
    # Low-E'' line weakens on heating, high-E'' line strengthens: no universal trend.
    low = [scale_intensity(1e-19, 2300.0, 0.0, T_REF / T, T) for T in (250, 296, 400)]
    high = [scale_intensity(1e-19, 2300.0, 1500.0, T_REF / T, T) for T in (250, 296, 400)]
    assert low[0] > low[1] > low[2]
    assert high[0] < high[1] < high[2]


def test_stimulated_emission_rayleigh_jeans_limit():
    rE = scale_intensity(1.0, 1e-4, 0.0, 1.0, 200.0)   # nu -> 0: ratio -> T_ref/T
    assert abs(rE - T_REF / 200.0) < 1e-6


# -------------------------------------------------------------- broadening
def test_pressure_and_temperature_dependence_of_lorentz_width():
    g = lorentz_hwhm(0.07, 0.09, 0.7, 296.0, 101325.0, 0.0)
    assert abs(g - 0.07) < 1e-12                           # 1 atm, 296 K, no self
    g = lorentz_hwhm(0.07, 0.09, 0.7, 296.0, 0.5 * 101325.0, 0.0)
    assert abs(g - 0.035) < 1e-12                          # linear in P
    g = lorentz_hwhm(0.07, 0.09, 0.7, 200.0, 101325.0, 0.0)
    assert abs(g - 0.07 * (296 / 200) ** 0.7) < 1e-12
    g = lorentz_hwhm(0.07, 0.09, 0.7, 296.0, 101325.0, 0.1)   # self part mixed in
    assert abs(g - (0.07 * 0.9 + 0.09 * 0.1)) < 1e-12


# ------------------------------------------------- units / sum rule / gas law
def test_loschmidt_number_density():
    n = number_density_cm3(273.15, 101325.0) * 1e6      # back to m^-3
    assert abs(n - 2.686780111e25) / 2.686780111e25 < 1e-6


def test_line_strength_sum_rule():
    # integral of sigma d nu over an isolated line must equal S(T) [cm/molecule]
    nu0, S0, W = 2300.0, 1e-18, 400.0
    grid = np.arange(nu0 - W, nu0 + W, 0.005)
    lines = make_lines([nu0], [S0], E=200.0)
    sig = species_cross_section(grid, lines, 250.0, 0.5 * 101325, 0.0, toy_Q, wing_cm1=W)
    S_T = scale_intensity(S0, nu0, 200.0, T_REF / 250.0, 250.0)
    gL = lorentz_hwhm(0.07, 0.07, 0.7, 250.0, 0.5 * 101325, 0.0)
    area = trapezoid(sig, grid) + S_T * 2 * gL / (np.pi * W)
    assert abs(area - S_T) / S_T < 2e-5


# -------------------------------------------------------- Beer-Lambert etc.
def _toy_mixture(x_co2=4e-4, x_n2o=3.4e-7, **kw):
    grid = np.arange(2290.0, 2310.0, 0.005)
    co2 = make_lines([2295.0, 2300.0, 2304.5], [3e-18, 1e-18, 2e-18], E=[50, 300, 900], M=2)
    n2o = make_lines([2299.0, 2302.0], [1e-18, 5e-19], E=[100, 400], M=4, mass=44.0)
    a1, a2 = absorption_coefficients(grid, co2, n2o, 255.0, 54020.0, x_co2, x_n2o, toy_Q,
                                        wing_cm1=10.0, **kw)
    return grid, a1, a2


def test_beer_lambert_consistency_and_bounds():
    grid, a1, a2 = _toy_mixture()
    I0 = np.linspace(1.0, 2.0, len(grid))
    tau, T, I = beer_lambert(a1 + a2, 1.0e4, I0)
    assert np.allclose(T, np.exp(-(a1 + a2) * 1.0e4))
    assert np.all((T >= 0) & (T <= 1))
    assert np.allclose(I, I0 * T)
    m = T > 0
    assert np.allclose(tau[m], -np.log(T[m]), rtol=1e-9, atol=1e-12)


def test_zero_absorber_gives_unit_transmission():
    grid, a1, a2 = _toy_mixture(x_co2=0.0, x_n2o=0.0)
    _, T, _ = beer_lambert(a1 + a2, 1e6, np.ones_like(grid))
    assert np.all(T == 1.0)


def test_low_abundance_and_short_path_limit():
    grid, a1, a2 = _toy_mixture(x_co2=1e-12, x_n2o=1e-12)
    tau, T, _ = beer_lambert(a1 + a2, 1.0, np.ones_like(grid))
    assert np.max(np.abs((1 - T) - tau)) < 1e-12           # 1 - exp(-tau) ~ tau


def test_optically_thick_limit():
    grid, a1, a2 = _toy_mixture(x_co2=0.2)
    _, T, _ = beer_lambert(a1 + a2, 1.0e6, np.ones_like(grid))
    assert T.min() < 1e-10


def test_mixture_additivity_and_linearity_in_mole_fraction():
    # with gamma_self == gamma_air the widths do not depend on x, so alpha is exactly linear in x
    g1, a1, _ = _toy_mixture(x_co2=2e-4)
    g2, a2, _ = _toy_mixture(x_co2=4e-4)
    assert np.allclose(2 * a1, a2, rtol=1e-12)
    # additivity: each gas computed ALONE must equal its component in the mixture,
    # so that alpha_total = alpha_CO2 + alpha_N2O with no cross-coupling in this model
    grid, c, n = _toy_mixture()
    _, c_alone, _ = _toy_mixture(x_n2o=0.0)
    _, _, n_alone = _toy_mixture(x_co2=0.0)
    assert np.allclose(c, c_alone, rtol=1e-12, atol=0)
    assert np.allclose(n, n_alone, rtol=1e-12, atol=0)
    tau_total = (c + n) * 1.0e4
    tau_sep = c * 1.0e4 + n * 1.0e4
    assert np.allclose(tau_total, tau_sep, rtol=1e-12, atol=0)


# --------------------------------------------------------- resolution tests
def test_resolution_convergence():
    rng = np.random.default_rng(1)
    nu = np.sort(rng.uniform(2295, 2305, 25))
    lines = make_lines(nu, rng.uniform(1e-19, 3e-18, 25), E=rng.uniform(0, 1500, 25))

    def integrated(step):
        grid = np.arange(2290.0, 2310.0, step)
        sig = species_cross_section(grid, lines, 255.0, 54020.0, 4e-4, toy_Q, wing_cm1=10.0)
        return trapezoid(sig, grid)
    ref = integrated(0.00125)
    err_fine = abs(integrated(0.005) - ref) / ref
    err_coarse = abs(integrated(0.1) - ref) / ref
    assert err_fine < 1e-4
    assert err_coarse > err_fine                           # error shrinks as resolution improves


# ----------------------------------------------------------- source and ISA
def test_planck_peak_follows_wien_law_in_wavenumber():
    T = 1500.0
    nu = np.arange(500.0, 8000.0, 0.5)
    peak = nu[np.argmax(planck_per_cm1(nu, T))]
    assert abs(peak - 2.8214393721 * T / C2_CM_K) < 1.0


def test_isa_troposphere_matches_tabulated_tropopause():
    T, P = isa_troposphere(11000.0)
    assert abs(T - 216.65) < 1e-9
    assert abs(P - 22632.1) / 22632.1 < 1e-3               # published table value
    T0, P0 = isa_troposphere(0.0)
    assert abs(T0 - 288.15) < 1e-12 and abs(P0 - 101325.0) < 1e-9


def test_dilution_warning():
    assert dilution_warnings(4e-4, 3e-7) == []
    assert len(dilution_warnings(0.2, 3e-7)) == 1


# ------------------------------------------- many frames from few exact ones
def _hard_lines(seed=3, n=150):
    # deliberately hard: huge spread of lower-state energies (up to 3000 cm^-1) and strengths
    rng = np.random.default_rng(seed)
    nu = np.sort(rng.uniform(2255.0, 2345.0, n))
    f = lambda v: np.full(n, v)
    return LineList("toy", 2, nu, 10 ** rng.uniform(-22, -18, n), rng.uniform(0, 3000, n),
                       rng.uniform(0.05, 0.09, n), f(0.09), rng.uniform(0.6, 0.8, n), f(-0.003),
                       np.ones(n, int), {1: 44.0})


def _exact_vs_interpolated(Ts, Ps):
    lines = _hard_lines()
    Q = lambda M, I, T: T ** 1.5
    grid = np.arange(2270.0, 2330.0, 0.005)
    one = lambda T, P: species_cross_section(grid, lines, T, P, 4e-4, Q, wing_cm1=10.0) * 4e-4 * number_density_cm3(T, P)
    exact = np.array([one(T, P) for T, P in zip(Ts, Ps)])
    anchors = choose_anchors(Ts, Ps)
    interp = interpolate_between_anchors(Ts, anchors, exact[anchors])
    core = exact > 1e-3 * exact.max()
    rel = np.max(np.abs(interp - exact)[core] / exact[core])
    dT = np.max(np.abs(np.exp(-interp * 1e4) - np.exp(-exact * 1e4)))      # 100 m path
    return anchors, rel, dT


def test_choose_anchors_spacing_and_ends():
    Ts = np.linspace(216.0, 316.0, 101)
    a = choose_anchors(Ts, np.full(101, 5e4))
    assert a[0] == 0 and a[-1] == 100
    assert np.max(np.diff(Ts[a])) <= 10.0 + 1e-9 and len(a) == 11
    Ts2, Ps2 = isa_troposphere(np.linspace(0, 11000, 101))
    o = np.argsort(Ts2)
    a2 = choose_anchors(Ts2[o], Ps2[o])
    assert np.max(np.abs(np.diff(np.log(Ps2[o][a2])))) <= 0.07 + 1e-9      # pressure spacing respected


def test_interpolated_frames_accuracy_fixed_pressure():
    Ts = np.linspace(216.0, 316.0, 101)
    anchors, rel, dT = _exact_vs_interpolated(Ts, np.full(101, 54019.9))
    assert len(anchors) == 11
    assert rel < 5e-3 and dT < 2e-3, (rel, dT)


def test_interpolated_frames_accuracy_isa_path():
    Ts, Ps = isa_troposphere(np.linspace(0, 11000, 101))
    o = np.argsort(Ts)
    anchors, rel, dT = _exact_vs_interpolated(Ts[o], Ps[o])
    assert rel < 1e-2 and dT < 4e-3, (rel, dT)   # residual dominated by pressure-shift/cut-off-edge effects


def test_anchor_frames_are_returned_unchanged():
    Ts = np.linspace(216.0, 316.0, 21)
    A = np.abs(np.random.default_rng(0).normal(size=(3, 50))) + 1e-3
    out = interpolate_between_anchors(Ts, np.array([0, 10, 20]), A)
    assert np.array_equal(out[[0, 10, 20]], A)
    assert np.all(out > 0)


def test_envelope_keeps_extremes_and_length():
    x = np.linspace(0.0, 10.0, 100000)
    y = np.sin(37 * x) * np.exp(-0.1 * x)
    xs, ys = envelope(x, y, 1500)
    assert len(xs) == 3000 and len(ys) == 3000
    assert ys.min() == y.min() and ys.max() == y.max()
    assert np.all(np.diff(xs) >= 0)


# ------------------------------------------------ wavelength-based quantities
def test_planck_per_um_follows_wien_law_in_wavelength():
    lam = np.linspace(0.5, 10.0, 200000)
    peak = lam[np.argmax(planck_per_um(lam, 1500.0))]
    assert abs(peak - 2897.771955 / 1500.0) < 1e-3          # Wien displacement constant b = 2897.771955 um K


def test_wavelength_and_wavenumber_forms_of_planck_carry_the_same_power():
    nu = np.arange(2250.0, 2350.0, 0.01)
    p_nu = trapezoid(planck_per_cm1(nu, 1500.0), nu)                      # W m^-2 sr^-1
    p_lam = integrate_dlambda_um(nu, planck_per_um(1.0e4 / nu, 1500.0))   # W m^-2 sr^-1
    assert abs(p_nu - p_lam) / p_nu < 1e-6


def test_integral_over_wavelength_gives_the_wavelength_range():
    nu = np.linspace(2250.0, 2350.0, 10001)                      # endpoint included
    area = integrate_dlambda_um(nu, np.ones_like(nu))
    assert abs(area - (1.0e4 / 2250.0 - 1.0e4 / 2350.0)) < 1e-6      # 4.4444 - 4.2553 um


def test_window_mean_of_constant_and_flat_incident_spectrum():
    nu = np.arange(2250.0, 2350.0, 0.01)
    assert abs(window_mean_um(nu, np.full_like(nu, 0.37)) - 0.37) < 1e-12
    I0, unit = make_incident_spectrum(nu, True, 1500.0)
    assert np.all(I0 == 1.0) and "FLAT" in unit


# ------------------------------------------------ constant-number-density sweep
def test_pressure_at_constant_density_is_n_k_T():
    Ts = np.linspace(195.0, 300.0, 50)
    Ps = pressure_at_constant_density(1.53e25, Ts)
    assert np.allclose(Ps, 1.53e25 * 1.380649e-23 * Ts, rtol=1e-14)
    n = np.array([number_density_cm3(T, P) for T, P in zip(Ts, Ps)]) * 1.0e6      # back to m^-3
    assert np.allclose(n, 1.53e25, rtol=1e-12)                    # density really is constant
    assert np.all(np.diff(Ps) > 0)                                 # pressure rises with temperature


def test_isochoric_and_isobaric_thin_line_trends_differ():
    # Optically thin line, Q ~ T: d ln W / d ln T = -1 + c2 E''/T (constant n) versus -2 + c2 E''/T (constant P).
    # Pick E'' = 260 cm^-1 at T ~ 250 K: constant n -> grows, constant P -> falls.
    def W(T, n_scales_with_T_inverse):
        S = scale_intensity(1.0, 2300.0, 260.0, T_REF / T, T)
        return S * (T_REF / T if n_scales_with_T_inverse else 1.0)
    iso = np.log(W(260.0, False) / W(240.0, False))
    isob = np.log(W(260.0, True) / W(240.0, True))
    assert iso > 0 > isob
    mid = np.log(260.0 / 240.0)
    assert abs(iso / mid - (-1 + C2_CM_K * 260.0 / 250.0)) < 0.03
    assert abs(isob / mid - (-2 + C2_CM_K * 260.0 / 250.0)) < 0.03


def test_interpolated_frames_accuracy_isochoric_path():
    Ts = np.linspace(195.0, 300.0, 96)
    Ps = pressure_at_constant_density(1.53e25, Ts)
    anchors, rel, dT = _exact_vs_interpolated(Ts, Ps)
    assert len(anchors) <= 14
    assert rel < 1e-2 and dT < 4e-3, (rel, dT)

In [18]:
# ===== NEW tests (all SYNTHETIC data). A test may return a short string with its key number; it is shown in the validation table. =====
import contextlib

@contextlib.contextmanager
def _tmpdir():
    d = tempfile.mkdtemp(prefix="co2n2o_test_")
    try:
        yield d
    finally:
        shutil.rmtree(d, ignore_errors=True)


@contextlib.contextmanager
def _stub_iso_table():
    """Replace the HAPI isotopologue table by a fixed one (tests only: no HAPI needed)."""
    old = globals()["iso_table"]
    globals()["iso_table"] = lambda M: {1: (1, "main", 0.99, 44.0), 2: (2, "minor", 0.01, 45.0)}
    try:
        yield
    finally:
        globals()["iso_table"] = old


@contextlib.contextmanager
def _cfg_override(**kw):
    old = {k: getattr(cfg, k) for k in kw}
    for k, v in kw.items():
        setattr(cfg, k, v)
    try:
        yield
    finally:
        for k, v in old.items():
            setattr(cfg, k, v)


# ------------------------------------------------------------- line-list integrity
def test_parser_reads_every_field_of_a_160_character_record():
    with _tmpdir() as d:
        p = os.path.join(d, "CO2_a.par")
        line = format_par_line(2, 1, 2349.143006, 3.5e-18, 420.0, 0.0750, 0.0900, 1234.5678, 0.69, -0.00312, quanta="00011   00001")
        assert len(line) == 160
        with open(p, "w") as fh:
            fh.write(line + "\n")
        recs, bad = read_par_file(p, 2)
        row = recs[0][2]
        assert bad == 0 and len(recs) == 1
        assert row["M"] == 2 and row["I"] == 1
        assert abs(row["nu"] - 2349.143006) < 1e-9 and abs(row["S"] - 3.5e-18) < 1e-22
        assert abs(row["gamma_air"] - 0.075) < 1e-9 and abs(row["gamma_self"] - 0.09) < 1e-9
        assert abs(row["E_low"] - 1234.5678) < 1e-9 and abs(row["n_air"] - 0.69) < 1e-9 and abs(row["delta_air"] + 0.00312) < 1e-9


def test_same_lines_in_data_and_par_files_are_not_double_counted():
    rows = [(1, 2300.0 + 0.7 * k, 1e-19 * (k + 1), 50.0 * k, "Q%d" % k) for k in range(20)]
    with _tmpdir() as d:
        write_par(os.path.join(d, "CO2_x.data"), rows)
        write_par(os.path.join(d, "CO2_x.par"), rows)
        paths = sorted(glob.glob(os.path.join(d, "CO2*.data")) + glob.glob(os.path.join(d, "CO2*.par")))
        cols, rep = parse_par_files(paths, 2)
        assert len(cols["nu"]) == 20 and rep["exact_duplicates_removed"] == 20, rep
    return "20 lines loaded from 2 identical files (20 duplicates removed)"


def test_physically_distinct_lines_are_never_removed():
    # same position and strength but different quantum numbers = different transitions; repeated records inside ONE file are kept
    rows = [(1, 2300.0, 1e-19, 50.0, "A"), (1, 2300.0, 1e-19, 50.0, "B"), (1, 2300.0, 1e-19, 50.0, "B")]
    with _tmpdir() as d:
        write_par(os.path.join(d, "CO2_1.par"), rows)
        cols, rep = parse_par_files([os.path.join(d, "CO2_1.par")], 2)
        assert len(cols["nu"]) == 3 and rep["exact_duplicates_removed"] == 0
        write_par(os.path.join(d, "CO2_2.data"), rows)                       # the same three again in a second file
        cols, rep = parse_par_files([os.path.join(d, "CO2_1.par"), os.path.join(d, "CO2_2.data")], 2)
        assert len(cols["nu"]) == 3 and rep["exact_duplicates_removed"] == 3


def test_same_transition_with_different_numbers_in_two_files_is_an_error():
    with _tmpdir() as d:
        write_par(os.path.join(d, "CO2_1.par"), [(1, 2300.0, 1e-19, 50.0, "A")])
        write_par(os.path.join(d, "CO2_2.par"), [(1, 2300.0, 2e-19, 50.0, "A")])
        try:
            parse_par_files(sorted(glob.glob(os.path.join(d, "*.par"))), 2)
        except ValueError:
            return
        raise AssertionError("conflicting editions should raise ValueError")


def test_load_linelist_merges_files_sorts_lines_and_ignores_header_files():
    rows = [(1, 2310.0, 1e-19, 50.0, "c"), (2, 2290.0, 2e-19, 60.0, "a"), (1, 2300.0, 3e-19, 70.0, "b")]
    with _tmpdir() as d, _stub_iso_table():
        write_par(os.path.join(d, "CO2_t.data"), rows)
        write_par(os.path.join(d, "CO2_t.par"), rows)
        with open(os.path.join(d, "CO2_t.header"), "w") as fh:
            fh.write('{"table_name": "CO2_t"}')
        lines, rep, paths = load_linelist("CO2", 2, d)
        assert len(lines.nu) == 3 and np.all(np.diff(lines.nu) > 0)
        assert not any(p.endswith(".header") for p in paths) and len(paths) == 2
        assert rep["duplicates"]["exact_duplicates_removed"] == 3
        assert lines.mass_amu == {1: 44.0, 2: 45.0}


def test_missing_broadening_parameters_are_replaced_and_counted():
    cols = dict(I=np.array([1, 1, 1, 2]), gamma_air=np.array([0.07, 0.0, 0.09, 0.08]), n_air=np.array([0.7, 0.7, -1.0, 0.6]),
                gamma_self=np.array([0.09, 0.09, 0.0, 0.1]))
    rep = repair_missing(cols)
    assert rep["gamma_air"] == 1 and rep["n_air"] == 1 and rep["gamma_self"] == 1
    assert abs(cols["gamma_air"][1] - 0.08) < 1e-12 and abs(cols["n_air"][2] - 0.7) < 1e-12
    assert cols["gamma_self"][2] == cols["gamma_air"][2]


# ------------------------------------------------------------ spectral coverage
def test_coverage_check_uses_the_recorded_download_range():
    co2, _ = synthetic_linelists()
    with _tmpdir() as d:
        mp = os.path.join(d, "data_manifest.json")
        assert check_coverage(co2, cfg.NU_MIN, cfg.NU_MAX, cfg.WING_CM1, mp)["confirmed"] is None      # no manifest: unconfirmed
        with open(mp, "w") as fh:
            json.dump({"wavenumber_range_cm-1": [cfg.NU_MIN - cfg.FETCH_MARGIN_CM1, cfg.NU_MAX + cfg.FETCH_MARGIN_CM1]}, fh)
        assert check_coverage(co2, cfg.NU_MIN, cfg.NU_MAX, cfg.WING_CM1, mp)["confirmed"] is True
        with open(mp, "w") as fh:
            json.dump({"wavenumber_range_cm-1": [cfg.NU_MIN, cfg.NU_MAX]}, fh)                      # window only, no wing margin
        assert check_coverage(co2, cfg.NU_MIN, cfg.NU_MAX, cfg.WING_CM1, mp)["confirmed"] is False


def test_band_edge_check_flags_a_band_that_the_window_cuts_through():
    co2, n2o = synthetic_linelists()
    assert check_band_edges(co2, cfg.NU_MIN, cfg.NU_MAX)["strong_lines_near_edges"] == 0
    assert check_band_edges(n2o, cfg.NU_MIN, cfg.NU_MAX)["strong_lines_near_edges"] == 0
    assert check_band_edges(co2, 2300.0, 2380.0)["strong_lines_near_edges"] > 0           # window inside the band


def test_default_window_contains_both_band_centres_and_the_download_covers_the_wings():
    assert cfg.NU_MIN < 2224.0 - 40 and cfg.NU_MAX > 2349.0 + 60            # both nu3 band centres well inside, R branch of CO2 included
    assert cfg.NU_MIN - cfg.FETCH_MARGIN_CM1 <= cfg.NU_MIN - cfg.WING_CM1
    assert cfg.NU_MAX + cfg.FETCH_MARGIN_CM1 >= cfg.NU_MAX + cfg.WING_CM1
    assert abs(cfg.LAM_MIN_UM - 1e4 / cfg.NU_MAX) < 1e-12 and abs(cfg.LAM_MAX_UM - 1e4 / cfg.NU_MIN) < 1e-12


def test_default_mole_fraction_is_gas_specific():
    # old bug: check_against_hapi() used the N2O abundance as default for CO2
    assert default_x("CO2") == cfg.X_CO2 and default_x("N2O") == cfg.X_N2O and default_x("CO2") != default_x("N2O")


# ------------------------------------------------------------ atmosphere table
def test_afgl_table_is_consistent_with_slider_ranges_and_the_ideal_gas_law():
    lo, hi = afgl_density_range_m3()
    assert abs(lo - 3.49e24) / 3.49e24 < 0.01 and abs(hi - 2.85e25) / 2.85e25 < 0.01        # numbers quoted in the study text
    assert 10 ** cfg.N_BOUNDS_LOG10_M3[0] <= lo and 10 ** cfg.N_BOUNDS_LOG10_M3[1] >= hi
    Ts = [r[2] for r in AFGL_REF]
    Ps = [r[3] * 100.0 for r in AFGL_REF]
    assert cfg.T_BOUNDS[0] <= min(Ts) and cfg.T_BOUNDS[1] >= max(Ts)
    assert cfg.P_BOUNDS[0] <= min(Ps) and cfg.P_BOUNDS[1] >= max(Ps)


def test_us_standard_rows_follow_isa_when_altitude_is_geometric():
    worst_T, worst_P = 0.0, 0.0
    for k in US_STD_ROWS:
        _, z, T, p = AFGL_REF[k]
        Ti, Pi = isa_troposphere_geometric(z * 1000.0)
        worst_T, worst_P = max(worst_T, abs(T - float(Ti))), max(worst_P, abs(p * 100.0 - float(Pi)) / float(Pi))
    assert worst_T < 0.1 and worst_P < 1e-3, (worst_T, worst_P)
    return "AFGL US-Std rows vs ISA(geometric): max |dT| = %.3f K, max |dP/P| = %.1e" % (worst_T, worst_P)


def test_geometric_and_geopotential_heights_differ_as_documented():
    assert abs(float(geopotential_from_geometric_m(11000.0)) - 10981.2) < 0.5
    z_trop = 1e-3 * R0_US1976_M * 11000.0 / (R0_US1976_M - 11000.0)        # geometric height of the geopotential 11 km tropopause
    assert abs(z_trop - 11.019) < 0.001
    assert abs(isa_troposphere_geometric(11000.0)[0] - 216.65) > 0.05      # NOT the tropopause temperature: 11 km geometric is 19 m below it


# ------------------------------------------------ line shape, cut-off, threshold
def test_wing_cutoff_removes_the_analytic_lorentzian_tail():
    nu0, S0, W = 2300.0, 1e-18, 20.0
    grid = np.arange(nu0 - 60, nu0 + 60, 0.0025)
    lines = make_lines([nu0], [S0])
    P = 0.5 * ATM_PA
    gL = lorentz_hwhm(0.07, 0.07, 0.7, 250.0, P, 0.0)
    S_T = scale_intensity(S0, nu0, 0.0, T_REF / 250.0, 250.0)
    hard = trapezoid(species_cross_section(grid, lines, 250.0, P, 0.0, toy_Q, wing_cm1=W), grid)
    soft = trapezoid(species_cross_section(grid, lines, 250.0, P, 0.0, toy_Q, wing_cm1=W, subtract_edge=True), grid)
    tail = S_T * 2 * gL / (np.pi * W)
    assert abs((S_T - hard) - tail) / tail < 0.03                              # missing area = 2 gL / (pi W) x S
    assert abs((S_T - soft) - 2 * tail) / (2 * tail) < 0.03                    # continuous cut-off removes about twice that
    return "missing line area: hard cut %.2e, continuous cut %.2e of S (analytic 2gL/(pi W) = %.2e)" % ((S_T - hard) / S_T, (S_T - soft) / S_T, tail / S_T)


def test_continuous_cutoff_has_no_steps_when_a_line_shifts_across_a_grid_point():
    grid = np.arange(2290.0, 2310.0, 0.0025)
    steps = {}
    for edge in (False, True):
        prev, worst = None, 0.0
        for shift in np.arange(0.0, 0.0030, 0.0003):
            sig = species_cross_section(grid, make_lines([2300.0 + shift], [1e-18]), 250.0, 0.5 * ATM_PA, 0.0, toy_Q,
                                        wing_cm1=5.0, subtract_edge=edge)
            far = np.abs(grid - 2300.0) > 4.9
            if prev is not None:
                worst = max(worst, float(np.max(np.abs(sig - prev)[far])))
            prev = sig
        steps[edge] = worst
    assert steps[False] > 0 and steps[True] < 0.01 * steps[False], steps
    return "largest step in sigma at the cut-off: hard %.2e, continuous %.2e cm^2" % (steps[False], steps[True])


def test_weak_line_threshold_error_equals_the_dropped_line_strength():
    rng = np.random.default_rng(5)
    nu = np.sort(rng.uniform(2295, 2305, 60))
    S = 10 ** rng.uniform(-24, -19, 60)
    lines = make_lines(nu, S, E=0.0)
    grid = np.arange(2290.0, 2310.0, 0.0025)
    kw = dict(wing_cm1=10.0, subtract_edge=False)
    full = species_cross_section(grid, lines, T_REF, 0.5 * ATM_PA, 0.0, toy_Q, s_min=0.0, **kw)
    cut = species_cross_section(grid, lines, T_REF, 0.5 * ATM_PA, 0.0, toy_Q, s_min=1e-21, **kw)
    dropped = S[S < 1e-21].sum() / S.sum()
    got = (trapezoid(full, grid) - trapezoid(cut, grid)) / trapezoid(full, grid)
    assert abs(got - dropped) < 0.01 * dropped + 1e-6, (got, dropped)


def test_threshold_is_applied_to_the_line_strength_at_the_gas_temperature():
    # a hot-band-like line with large E'' is much weaker at 200 K than at 296 K and must be dropped by S(T), not by S(296 K)
    lines = make_lines([2300.0], [2e-26], E=2000.0)
    grid = np.arange(2290.0, 2310.0, 0.005)
    assert species_cross_section(grid, lines, 296.0, 0.5 * ATM_PA, 0.0, toy_Q, s_min=1e-26).max() > 0
    assert species_cross_section(grid, lines, 200.0, 0.5 * ATM_PA, 0.0, toy_Q, s_min=1e-26).max() == 0


def test_doppler_limit_peak_of_a_line_at_very_low_pressure():
    nu0, S0, T = 2300.0, 1e-18, 250.0
    grid = np.arange(nu0 - 0.05, nu0 + 0.05, 0.0001)
    sig = species_cross_section(grid, make_lines([nu0], [S0]), T, 1e-3, 0.0, toy_Q, wing_cm1=0.05)
    sigma_G = float(doppler_hwhm(nu0, T, 44.0)) / np.sqrt(2 * np.log(2))
    S_T = scale_intensity(S0, nu0, 0.0, T_REF / T, T)
    assert abs(sig.max() - S_T / (sigma_G * np.sqrt(2 * np.pi))) / sig.max() < 2e-3


# ------------------------------------------------- Beer-Lambert limits and units
def test_thin_line_equivalent_width_equals_line_strength_times_column():
    nu0, S0, W, T, P = 2300.0, 1e-18, 100.0, 250.0, 0.5 * ATM_PA
    grid = np.arange(nu0 - W, nu0 + W, 0.005)
    lines = make_lines([nu0], [S0], E=200.0)
    sig = species_cross_section(grid, lines, T, P, 0.0, toy_Q, wing_cm1=W)
    S_T = scale_intensity(S0, nu0, 200.0, T_REF / T, T)
    gL = lorentz_hwhm(0.07, 0.07, 0.7, T, P, 0.0)
    N = 1e-5 / (S_T / (np.pi * gL))                                      # column density giving peak tau = 1e-5 (optically thin)
    ew = trapezoid(1 - np.exp(-N * sig), grid) + N * S_T * 2 * gL / (np.pi * W)
    assert abs(ew - S_T * N) / (S_T * N) < 1e-3


def test_strongly_saturated_line_follows_the_square_root_law():
    # Lorentz-dominated, flat I0: W = 2 sqrt(S N gamma_L)  (HWHM gamma_L, column N); peak tau = S N / (pi gamma_L) = 3000 here
    nu0, S0, W, T, P = 2300.0, 1e-18, 400.0, 296.0, 0.25 * ATM_PA
    grid = np.arange(nu0 - W, nu0 + W, 0.005)
    sig = species_cross_section(grid, make_lines([nu0], [S0]), T, P, 0.0, toy_Q, wing_cm1=W)
    gL = lorentz_hwhm(0.07, 0.07, 0.7, T, P, 0.0)
    N = 3000.0 * np.pi * gL / S0
    ew = trapezoid(1 - np.exp(-N * sig), grid)
    expected = 2 * np.sqrt(S0 * N * gL)
    assert abs(ew - expected) / expected < 0.02, (ew, expected)
    return "W = %.4f cm-1 vs 2 sqrt(S N gL) = %.4f cm-1 (%.2f %% difference)" % (ew, expected, 100 * (ew - expected) / expected)


def test_transmittance_of_two_layers_multiplies():
    grid, a1, a2 = _toy_mixture()
    _, T1, _ = beer_lambert(a1 + a2, 3.0e3, np.ones_like(grid))
    _, T2, _ = beer_lambert(a1 + a2, 7.0e3, np.ones_like(grid))
    _, T12, _ = beer_lambert(a1 + a2, 1.0e4, np.ones_like(grid))
    assert np.allclose(T1 * T2, T12, rtol=1e-12, atol=1e-300)


def test_absorption_coefficient_agrees_with_an_independent_si_calculation():
    # absorption coefficient at a line peak recomputed in SI units (m, m^-3): alpha[m^-1] = n[m^-3] * x * S[m/molecule] * phi[m]
    nu0, S0, T, P, x = 2300.0, 2e-19, 255.0, 54020.0, 4.23e-4
    grid = np.arange(nu0 - 1.0, nu0 + 1.0, 0.0005)
    a_cm = x * number_density_cm3(T, P) * species_cross_section(grid, make_lines([nu0], [S0], g_air=0.07, g_self=0.07), T, P, x, toy_Q, wing_cm1=1.0)
    S_T = scale_intensity(S0, nu0, 0.0, T_REF / T, T)
    gD = float(doppler_hwhm(nu0, T, 44.0))
    gL = lorentz_hwhm(0.07, 0.07, 0.7, T, P, x)
    phi_cm = float(voigt_profile(0.0, gD, gL))
    alpha_si = (P / (K_B * T)) * x * (S_T * 1e-2) * (phi_cm * 1e-2)
    assert abs(a_cm.max() * 100.0 - alpha_si) / alpha_si < 1e-3


def test_wavelength_wavenumber_conversion_round_trip():
    nu = np.array([2150.0, 2224.0, 2349.0, 2430.0])
    lam = 1e4 / nu
    assert np.allclose(1e4 / lam, nu, rtol=1e-14)
    assert abs(lam[2] - 4.2571) < 1e-3 and np.all(np.diff(lam) < 0)         # CO2 nu3 band centre is at about 4.257 um; lambda falls as nu rises


# ---------------------------------------- display binning and mole-fraction scaling
def test_bin_stats_uses_fixed_bins_that_do_not_depend_on_the_data():
    x = np.linspace(2150.0, 2430.0, 100001)
    y1, y2 = np.sin(30 * x), np.cos(7 * x) ** 3
    a, b = bin_stats(x, y1, 1400), bin_stats(x, y2, 1400)
    assert np.array_equal(a[0], b[0]) and len(a[0]) <= 1400                    # identical x positions for different data
    assert abs(a[1].min() - y1.min()) < 1e-15 and abs(a[2].max() - y1.max()) < 1e-15
    assert np.all(a[1] <= a[3] + 1e-15) and np.all(a[3] <= a[2] + 1e-15)


def test_mole_fraction_scaling_error_is_small():
    co2, n2o = synthetic_linelists()
    grid = np.arange(2335.0, 2345.0, 0.0025)
    e_rel, e_T = mole_fraction_scaling_error(co2, n2o, toy_Q, grid, 255.0, 54020.0, 423e-6, 846e-6, "CO2", 100.0)
    assert e_rel < 5e-3 and e_T < 5e-3, (e_rel, e_T)
    return "rescaling alpha_CO2 from 423 to 846 ppm (gamma_self = 1.2 gamma_air): max rel. error %.1e, max |dTransmittance| %.1e" % (e_rel, e_T)


# ---------------------------------------------------- temperature interpolation
def _interp_setup(window=(2290.0, 2360.0)):
    co2, n2o = synthetic_linelists()
    return co2, n2o, np.arange(window[0], window[1], cfg.NU_STEP)


def test_interpolation_error_between_anchors_is_small_isochoric():
    co2, n2o, grid = _interp_setup()
    r = interpolation_validation(co2, n2o, toy_Q, grid, "isochoric", 1.53e25, 0.0, 195.0, 300.0, 423e-6, 340e-9,
                                 dT_list=(10.0,), verbose=False)
    err_T, err_a = r[10.0]
    assert err_T < 5e-4 and err_a < 5e-3, r
    return "anchors every 10 K, worst-case mid-points: max |dTransmittance| = %.1e, max rel. d alpha = %.1e" % (err_T, err_a)


def test_interpolation_error_between_anchors_is_small_isobaric():
    co2, n2o, grid = _interp_setup()
    r = interpolation_validation(co2, n2o, toy_Q, grid, "isobaric", 0.0, 54000.0, 195.0, 300.0, 423e-6, 340e-9,
                                 dT_list=(10.0,), verbose=False)
    err_T, err_a = r[10.0]
    assert err_T < 5e-4 and err_a < 5e-3, r
    return "anchors every 10 K, worst-case mid-points: max |dTransmittance| = %.1e, max rel. d alpha = %.1e" % (err_T, err_a)


def test_conv_interpolation_error_falls_as_anchor_spacing_shrinks():
    co2, n2o, grid = _interp_setup((2300.0, 2330.0))
    with _cfg_override(S_MIN=0.0):                                          # remove the (separate) threshold noise floor
        r = interpolation_validation(co2, n2o, toy_Q, grid, "isobaric", 0.0, 54000.0, 195.0, 300.0, 423e-6, 340e-9,
                                     dT_list=(20.0, 10.0, 5.0), verbose=False)
    a20, a10, a5 = r[20.0][1], r[10.0][1], r[5.0][1]
    assert a10 < a20 / 3 and a5 < a10 / 3, r
    return "max rel. d alpha at mid-points: 20 K %.1e, 10 K %.1e, 5 K %.1e" % (a20, a10, a5)


def test_spectrum_is_continuous_in_temperature_across_an_anchor():
    co2, n2o, grid = _interp_setup((2300.0, 2330.0))
    mod = build_sweep_model(grid, co2, n2o, "isochoric", 1.53e25, 0.0, 195.0, 300.0, 423e-6, 340e-9, toy_Q, verbose=False)
    Ta = mod.T_anchor[3]
    f1, f2 = mod.frame(Ta - 1e-4), mod.frame(Ta + 1e-4)
    t1, t2 = np.exp(-(f1["a_co2"] + f1["a_n2o"]) * 1e6), np.exp(-(f2["a_co2"] + f2["a_n2o"]) * 1e6)
    assert np.max(np.abs(t1 - t2)) < 1e-5 and mod.frame(Ta)["exact"] and not f1["exact"]
    return "max |dTransmittance| across an anchor (+-1e-4 K, L = 10 km): %.1e" % np.max(np.abs(t1 - t2))


def test_sweep_model_composition_and_path_length_scaling_are_exact_where_claimed():
    co2, n2o, grid = _interp_setup((2300.0, 2330.0))
    mod = build_sweep_model(grid, co2, n2o, "isobaric", 0.0, 54000.0, 195.0, 300.0, 423e-6, 340e-9, toy_Q, verbose=False)
    f = mod.frame(250.0)
    g = mod.frame(250.0, x_co2=2 * 423e-6, x_n2o=0.5 * 340e-9)
    assert np.allclose(g["a_co2"], 2 * f["a_co2"], rtol=1e-12) and np.allclose(g["a_n2o"], 0.5 * f["a_n2o"], rtol=1e-12)
    assert mod.frame(1e9)["T"] == 300.0 and mod.frame(-5.0)["T"] == 195.0       # clipped to the computed range


# ---------------------------------------------- numerical convergence (synthetic lines)
def test_conv_grid_wing_and_threshold_study_on_synthetic_lines():
    co2, n2o = synthetic_linelists()
    worst = dict(base_dT=0.0, base_ew=0.0, step_effect=0.0)
    conds = [(296.0, 101325.0), (255.0, 54020.0), (190.0, 9000.0)]
    for win in ((2335.0, 2345.0), (2215.0, 2225.0)):
        res = convergence_study(co2, n2o, toy_Q, win, conds, L_m_list=(1.0, 100.0), verbose=False)
        for (name, T, L), (dT, dEW, dmean, da) in res.items():
            if name == "baseline":
                worst["base_dT"], worst["base_ew"] = max(worst["base_dT"], dT), max(worst["base_ew"], dEW)
                step4 = res[("step x4", T, L)]
                worst["step_effect"] = max(worst["step_effect"], abs(step4[0] - dT))
    assert worst["base_dT"] < 1e-2 and worst["base_ew"] < 1e-2 and worst["step_effect"] < 1e-5, worst
    return ("baseline (step %.4g, wing %.0f cm-1) vs reference: max |dTransmittance| %.1e, max rel. error of equivalent width %.1e; "
            "going from step 4x coarser to baseline changes |dTransmittance| by < %.0e"
            % (cfg.NU_STEP, cfg.WING_CM1, worst["base_dT"], worst["base_ew"], max(worst["step_effect"], 1e-12)))

In [19]:
def run_tests(verbose=True):
    """Run every test_* function (the 31 original synthetic tests + the new ones) and fill the validation registry."""
    names = sorted(n for n, f in globals().items() if n.startswith("test_") and callable(f))
    failed = 0
    for n in names:
        cat = CAT_CONV if n.startswith("test_conv_") else CAT_SYN
        t0 = time.time()
        try:
            metric = globals()[n]()
            record(cat, n[5:], "PASS", metric if isinstance(metric, str) else "")
            if verbose:
                print("PASS", n, "(%.1fs)" % (time.time() - t0))
        except Exception as exc:
            failed += 1
            record(cat, n[5:], "FAIL", repr(exc)[:150])
            print("FAIL", n, "->", repr(exc))
    print("\n%d/%d tests passed (all on SYNTHETIC data: they test equations and code, not HITRAN)" % (len(names) - failed, len(names)))

run_tests()

PASS test_absorption_coefficient_agrees_with_an_independent_si_calculation (0.0s)
PASS test_afgl_table_is_consistent_with_slider_ranges_and_the_ideal_gas_law (0.0s)
PASS test_anchor_frames_are_returned_unchanged (0.0s)
PASS test_band_edge_check_flags_a_band_that_the_window_cuts_through (0.0s)
PASS test_beer_lambert_consistency_and_bounds (0.0s)
PASS test_bin_stats_uses_fixed_bins_that_do_not_depend_on_the_data (0.0s)
PASS test_c2_value (0.0s)
PASS test_choose_anchors_spacing_and_ends (0.0s)
PASS test_continuous_cutoff_has_no_steps_when_a_line_shifts_across_a_grid_point (0.0s)
PASS test_conv_grid_wing_and_threshold_study_on_synthetic_lines (2.1s)
PASS test_conv_interpolation_error_falls_as_anchor_spacing_shrinks (3.3s)
PASS test_coverage_check_uses_the_recorded_download_range (0.0s)
PASS test_default_mole_fraction_is_gas_specific (0.0s)
PASS test_default_window_contains_both_band_centres_and_the_download_covers_the_wings (0.0s)
PASS test_dilution_warning (0.0s)
PASS test_doppler_limit_p

## 7. Load data and checks

Run `fetch_hitran()` once (or place `.par` files in `data/`), then load. Without data the cell stops with instructions; set `cfg.ALLOW_SYNTHETIC_DEMO = True` only to practise with the widgets (the spectra are then invented and clearly labelled).

In [21]:
fetch_hitran()      # <- uncomment for the first run (needs your HITRAN account/API key set up for HAPI)

Using data

CO2_2155_2425
                     Lines parsed: 73495
N2O_2155_2425
                     Lines parsed: 43399

Data is fetched from http://hitran.org

http://hitran.org/lbl/api?iso_ids_list=7,8,9,10,11,12,13,14,121,15,120,122&numin=2125.0&numax=2455.0

BEGIN DOWNLOAD: CO2_2125_2455
  65536 bytes written to data/CO2_2125_2455.data
  65536 bytes written to data/CO2_2125_2455.data
  65536 bytes written to data/CO2_2125_2455.data
  65536 bytes written to data/CO2_2125_2455.data
  65536 bytes written to data/CO2_2125_2455.data
  65536 bytes written to data/CO2_2125_2455.data
  65536 bytes written to data/CO2_2125_2455.data
  65536 bytes written to data/CO2_2125_2455.data
  65536 bytes written to data/CO2_2125_2455.data
  65536 bytes written to data/CO2_2125_2455.data
  65536 bytes written to data/CO2_2125_2455.data
  65536 bytes written to data/CO2_2125_2455.data
  65536 bytes written to data/CO2_2125_2455.data
  65536 bytes written to data/CO2_2125_2455.data
  65536 bytes writt

UnboundLocalError: cannot access local variable 'dup' where it is not associated with a value

In [30]:
try:
    CO2, rep_c, files_c = load_linelist("CO2", 2, cfg.DATA_DIR)
    N2O, rep_n, files_n = load_linelist("N2O", 4, cfg.DATA_DIR)
    PARTITION_FN, DATA_LABEL = hapi_partition_fn, "HITRAN line data"
    for nm, L, rp, fl in (("CO2", CO2, rep_c, files_c), ("N2O", N2O, rep_n, files_n)):
        dup = rp.pop("duplicates")
        print("%s: %d lines from %s | exact duplicates removed: %d | parameters replaced by data-derived estimates: %s"
              % (nm, len(L.nu), [os.path.basename(p) for p in fl], dup["exact_duplicates_removed"], rp))
        cov = check_coverage(L, cfg.NU_MIN, cfg.NU_MAX, cfg.WING_CM1, os.path.join(cfg.DATA_DIR, "data_manifest.json"))
        print("   lines span %.1f-%.1f cm-1, needed %.0f-%.0f; coverage %s" % (cov["first_line"], cov["last_line"], *cov["need"],
              {True: "CONFIRMED by data_manifest.json", False: "INSUFFICIENT: re-download a wider range", None: "unconfirmed (no data_manifest.json: make sure your download covers the needed range)"}[cov["confirmed"]]))
except FileNotFoundError as exc:
    if not cfg.ALLOW_SYNTHETIC_DEMO:
        raise FileNotFoundError(str(exc) + "\n\nTo practise with the widgets WITHOUT HITRAN data set cfg.ALLOW_SYNTHETIC_DEMO = True "
                                           "(the spectra are then INVENTED and not physical).")
    CO2, N2O = synthetic_linelists()
    PARTITION_FN, DATA_LABEL = toy_Q, "SYNTHETIC DEMO - NOT HITRAN"
    print("*" * 100 + "\n*** SYNTHETIC DEMO: invented lines, toy partition function. NOT HITRAN, NOT physical. ***\n" + "*" * 100)
for msg in dilution_warnings(cfg.X_CO2, cfg.X_N2O):
    print(msg)
print("window: %.0f-%.0f cm-1 = %.4f-%.4f um" % (cfg.NU_MIN, cfg.NU_MAX, cfg.LAM_MIN_UM, cfg.LAM_MAX_UM))

CO2: 79317 lines from ['CO2_2125_2455.data', 'CO2_2155_2425.data'] | exact duplicates removed: 73495 | parameters replaced by data-derived estimates: {'gamma_air': 0, 'n_air': 0, 'gamma_self': 0}
   lines span 2125.0-2455.0 cm-1, needed 2130-2450; coverage INSUFFICIENT: re-download a wider range
N2O: 60423 lines from ['N2O_2125_2455.data', 'N2O_2155_2425.data'] | exact duplicates removed: 43399 | parameters replaced by data-derived estimates: {'gamma_air': 0, 'n_air': 0, 'gamma_self': 0}
   lines span 2125.0-2455.0 cm-1, needed 2130-2450; coverage INSUFFICIENT: re-download a wider range
window: 2150-2430 cm-1 = 4.1152-4.6512 um


In [32]:
def window_content():
    # How much of each gas's line strength lies in the window? (N2O's nu3 band is centred near 2224 cm-1.)
    print("Sum of line strengths S [cm/molecule at 296 K] and x_i * sum(S) (a rough absorption potential):")
    for lines, x in ((CO2, cfg.X_CO2), (N2O, cfg.X_N2O)):
        inw = (lines.nu >= cfg.NU_MIN) & (lines.nu <= cfg.NU_MAX)
        below = lines.nu < cfg.NU_MIN
        print("  %s: %5d lines in window, sum S = %.3e, x*sum S = %.3e | %d lines below the window (download margin), sum S = %.3e"
              % (lines.name, inw.sum(), lines.S[inw].sum(), x * lines.S[inw].sum(), below.sum(), lines.S[below].sum()))
    print("If N2O's sum S below the window is much larger than inside it, the window cuts through the N2O band: "
          "widen cfg.NU_MIN / cfg.NU_MAX (and the download) to include it.")

window_content()

Sum of line strengths S [cm/molecule at 296 K] and x_i * sum(S) (a rough absorption potential):
  CO2: 74316 lines in window, sum S = 1.012e-16, x*sum S = 4.282e-20 | 3473 lines below the window (download margin), sum S = 1.381e-22
  N2O: 46454 lines in window, sum S = 5.677e-17, x*sum S = 1.929e-23 | 8394 lines below the window (download margin), sum S = 3.182e-20
If N2O's sum S below the window is much larger than inside it, the window cuts through the N2O band: widen cfg.NU_MIN / cfg.NU_MAX (and the download) to include it.


In [34]:
atmosphere_check()
print()
study_table(CO2)
print()
window_content()
for L in (CO2, N2O):
    print(L.name, "band-edge check:", check_band_edges(L, cfg.NU_MIN, cfg.NU_MAX))

reference point (geometric height)         z/km      T/K      P/Pa       n/m^-3 |  ISA dT/K  ISA dP/%
Tropical, surface                             0    299.7    101300    2.448e+25 | 
Tropical, 17 km (tropical tropopause)        17    194.8      9370    3.484e+24 | 
Mid-latitude winter, surface                  0    272.2    101800    2.709e+25 | 
Subarctic winter, surface                     0    257.2    101300    2.853e+25 | 
U.S. Standard, surface                        0    288.2    101300    2.546e+25 |      0.05     -0.02
U.S. Standard, 5 km                           5    255.7     54050    1.531e+25 |      0.02      0.00
U.S. Standard, 11 km (geometric)             11    216.8     22700    7.584e+24 |      0.03      0.00
density range of the table: 3.484e+24 - 2.853e+25 m^-3
11 km GEOMETRIC = 10.981 km geopotential;  the ISA tropopause (216.65 K, 22632 Pa) is at 11 km GEOPOTENTIAL = 11.019 km geometric
U.S. Standard 1 km up: dT = 6.5 K, dP/P = 11.3 %  -> uniform-slab limit for

## 8. Interactive animation

Pick the **sweep mode** (constant number density, default, or constant pressure), then press **Compute / update**. The notebook computes exact spectra at *anchor* temperatures (at most 10 K apart); everything in between is a validated spline interpolation (error measured in §6/§9; the title says *exact line-by-line* or *interpolated, validated*).

* **Continuous temperature.** The temperature slider is continuous (0.1 K steps); **Play** sweeps the whole computed range (211 positions, 0.5 K apart for the default range).
* **No flicker.** The picture is a single image widget that is replaced in place (the cell is never cleared). All axes, the incident spectrum and the plot bins are fixed for the whole animation. Speed is limited by rendering (about a tenth of a second per frame, measured in the smoke test; slower on slow machines); for a fixed-speed animation use **Render movie**.
* **Instant controls:** temperature, path length (exact: τ = αL), CO₂ and N₂O amounts (α scaled linearly with x; exact except for the tiny x-dependence of the self-broadening width, measured by `test_mole_fraction_scaling_error_is_small`; the title says when α is rescaled), incident spectrum, selected wavelength, zoom window.
* **Need Compute:** sweep mode, number density or pressure, temperature range (line widths change). A few tens of seconds to minutes depending on the line list.
* **Incident vs transmitted:** bottom-left panel: grey = incident $I_0$ (constant), red = transmitted $I=I_0\mathcal T$. Drawing uses fixed bins: the band is the min–max inside a bin, the line the bin mean (a lower-resolution view). **Use the zoom slider** to resolve individual lines.

*If widgets do not render* (`ipywidgets>=8`, `jupyterlab_widgets`, restart JupyterLab) use `draw(T, L_m, "blackbody" or "flat", lambda_um)` after the first compute.

In [57]:
# The figure, the flicker-free live view and the pre-rendered movie
class Dashboard:
    """Three panels with FIXED axes: (top) transmittance, (bottom left) incident vs transmitted spectral intensity,
    (bottom right) absorption coefficients. All curves are drawn from FIXED horizontal bins (min/max band + bin mean),
    so successive frames line up exactly: no jitter, no jumping axes."""

    def __init__(self, grid, model, src="blackbody", view=None, figsize=(12.5, 7.8), dpi=120):
        self.grid, self.model, self.dpi = grid, model, dpi
        nu_lo, nu_hi = (grid[0], grid[-1]) if view is None else view
        i0, i1 = np.searchsorted(grid, nu_lo), np.searchsorted(grid, nu_hi, side="right")
        self.sl = slice(i0, max(i1, i0 + 2))
        self.nu = grid[self.sl]
        self.lam_min, self.lam_max = 1e4 / self.nu[-1], 1e4 / self.nu[0]
        self._w = 1.0e4 / grid ** 2
        self._wsum = trapezoid(self._w, grid)
        self.src = src
        self.fig = plt.figure(figsize=figsize, dpi=dpi)
        plt.close(self.fig)
        gs = self.fig.add_gridspec(2, 2, height_ratios=[1.35, 1], left=0.07, right=0.98, top=0.84, bottom=0.08, hspace=0.42, wspace=0.24)
        self.axT, self.axB, self.axD = (self.fig.add_subplot(gs[0, :]), self.fig.add_subplot(gs[1, 0]), self.fig.add_subplot(gs[1, 1]))
        self.title = self.fig.suptitle("", fontsize=8.5, family="monospace")
        self.I0, self.unit = make_incident_spectrum(grid, src == "flat", model.t_source)
        self.xc = None
        self._bands = []
        self._init_axes()

    def _init_axes(self):
        lam = np.zeros(2)
        for a in (self.axT, self.axB, self.axD):
            a.set_xlim(self.lam_min, self.lam_max)               # fixed for the whole animation
            a.set_xlabel("wavelength ($\\mu$m)", fontsize=9)
            a.xaxis.set_major_formatter(FormatStrFormatter("%.3f"))
            a.tick_params(labelsize=8)
        self.axT.set_ylim(0, 1.02)
        self.axT.set_ylabel("transmittance (dimensionless)", fontsize=9)
        self.axT.set_title("Transmittance  $\\mathcal{T}(\\lambda)=I/I_0=e^{-\\tau}$   (band = min-max inside each plot bin, black = bin mean, "
                           "green dotted = selected wavelength)", fontsize=9)
        (self.tCold,) = self.axT.plot(lam, lam, color="#7aa6d6", lw=0.8)
        (self.tHot,) = self.axT.plot(lam, lam, color="#e6a07a", lw=0.8)
        (self.tMean,) = self.axT.plot(lam, lam, color="crimson", lw=1.0)
        self.vT = self.axT.axvline(self.lam_min, color="g", ls=":", lw=1)
        self.axB.set_ylim(0, 1.05 * self.I0[self.sl].max())
        self.axB.set_ylabel(self.unit, fontsize=8)
        self.axB.set_title("Incident $I_0$ (fixed, grey) and transmitted $I=I_0\\mathcal{T}$ (red)", fontsize=9)
        (self.iInc,) = self.axB.plot(lam, lam, color="0.55", lw=1.4, label="incident $I_0$ (constant)")
        (self.iTr,) = self.axB.plot(lam, lam, color="C3", lw=0.9, label="transmitted $I$ (bin mean)")
        self.axB.legend(fontsize=7, loc="lower left")
        lo, hi = self.model.ylimD
        self.axD.set_yscale("log")
        self.axD.set_ylim(lo, hi)
        self.axD.set_ylabel("absorption coefficient (cm$^{-1}$)", fontsize=8)
        self.axD.set_title("Absorption coefficients (peak in each bin)", fontsize=9)
        (self.aCO2,) = self.axD.plot(lam, lam + 1, color="C1", lw=0.8, label="$\\alpha_{CO_2}$")
        (self.aN2O,) = self.axD.plot(lam, lam + 1, color="C2", lw=0.8, label="$\\alpha_{N_2O}$")
        (self.aTot,) = self.axD.plot(lam, lam + 1, color="crimson", lw=0.7, label="$\\alpha_{CO_2}+\\alpha_{N_2O}$ (additive)")
        self.vD = self.axD.axvline(self.lam_min, color="g", ls=":", lw=1)
        self.axD.legend(fontsize=7, loc="upper right")
        self._ref_key = None

    def _bins(self, y):
        return bin_stats(self.nu, y[self.sl], cfg.VIEW_BINS)

    def _band(self, ax, xc, ymin, ymax, color):
        self._bands.append(ax.fill_between(xc, ymin, ymax, color=color, alpha=0.35, lw=0))

    def render(self, T, L_m, x_co2, x_n2o, lam_sel_um):
        m = self.model
        fr = m.frame(T, x_co2, x_n2o)
        L_cm = L_m * 100.0
        tot = fr["a_co2"] + fr["a_n2o"]
        tau, trans, I = beer_lambert(tot, L_cm, self.I0)
        for b in self._bands:
            b.remove()
        self._bands = []
        xn, tmin, tmax, tmean = self._bins(trans)
        lam = 1e4 / xn
        self.tMean.set_data(lam, tmean)
        self._band(self.axT, lam, tmin, tmax, "#555555")
        key = (round(L_m, 9), x_co2, x_n2o)                              # coldest / warmest reference curves
        if key != self._ref_key:
            for line, Tx in ((self.tCold, m.T_lo), (self.tHot, m.T_hi)):
                f = m.frame(Tx, x_co2, x_n2o)
                line.set_data(lam, self._bins(np.exp(-(f["a_co2"] + f["a_n2o"]) * L_cm))[3])
            self._ref_key = key
        self.axT.legend(handles=[self.tCold, self.tHot, self.tMean], labels=["coldest (%.0f K)" % m.T_lo, "warmest (%.0f K)" % m.T_hi,
                        "current (%.1f K)" % fr["T"]], fontsize=8, loc="lower left", ncol=3)
        _, _, _, i0m = self._bins(self.I0)
        self.iInc.set_data(lam, i0m)
        _, imin, imax, imean = self._bins(I)
        self.iTr.set_data(lam, imean)
        self._band(self.axB, lam, imin, imax, "C3")
        for line, a in ((self.aCO2, fr["a_co2"]), (self.aN2O, fr["a_n2o"]), (self.aTot, tot)):
            line.set_data(lam, np.maximum(self._bins(a)[2], m.ylimD[0]))
        i_sel = int(np.argmin(np.abs(self.grid - 1e4 / lam_sel_um)))
        lam_marker = 1e4 / self.grid[i_sel]
        for v in (self.vT, self.vD):
            v.set_xdata([lam_marker, lam_marker])
        W_nm = 1000.0 * integrate_dlambda_um(self.grid, 1.0 - trans)
        t_mean = trapezoid(trans * self._w, self.grid) / self._wsum
        scaled = (abs(x_co2 / m.x_co2_ref - 1) > 1e-9) or (abs(x_n2o / m.x_n2o_ref - 1) > 1e-9)
        self.title.set_text(
            "CO2 + N2O slab, %.4f-%.4f um (%.1f-%.1f cm-1 shown)   [%s]   incident: %s\n"
            "T = %.1f K (%s)   P = %.0f Pa   n = %.3e m-3 (%s)   L = %.3g m   x_CO2 = %.2f ppm   x_N2O = %.1f ppb%s\n"
            "selected %.4f um: tau = %.3e, transmittance = %.3e   window-mean = %.4f   equivalent width = %.3f nm (flat-I0 measure)"
            % (self.lam_min, self.lam_max, self.nu[0], self.nu[-1], m.label, "blackbody %.0f K" % m.t_source if self.src != "flat" else "flat I0 = 1",
               fr["T"], "exact line-by-line" if fr["exact"] else "interpolated, validated", fr["P"], fr["P"] / (K_B * fr["T"]),
               "constant n" if m.mode == "isochoric" else "n changes with T", L_m, x_co2 * 1e6, x_n2o * 1e9,
               "  [alpha rescaled from the computed x]" if scaled else "", lam_marker, tau[i_sel], trans[i_sel], t_mean, W_nm))
        return fr

    def png_bytes(self):
        buf = io.BytesIO()
        self.fig.savefig(buf, format="png", dpi=self.dpi)         # no bbox_inches: constant image size, no jumping
        return buf.getvalue()


def build_animation(grid, model, L_m, x_co2, x_n2o, src="blackbody", lam_sel_um=None, view=None, n_frames=None, fps=None, dpi=70):
    """matplotlib FuncAnimation: n_frames temperatures evenly spaced over the computed range (default 96 frames at 24 fps = 4.0 s)."""
    import matplotlib.animation as animation
    n_frames, fps = n_frames or cfg.N_FRAMES, fps or cfg.FPS
    lam_sel_um = model.lam_sel_um if lam_sel_um is None else lam_sel_um
    d = Dashboard(grid, model, src, view, figsize=(12.5, 7.8), dpi=dpi)
    Ts = model.temperatures(n_frames)

    def _frame(k):
        d.render(Ts[k], L_m, x_co2, x_n2o, lam_sel_um)
        return []
    return animation.FuncAnimation(d.fig, _frame, frames=n_frames, interval=1000.0 / fps, blit=False, repeat=True)


def player_html(anim, fps=24):
    """Embeddable HTML player for `anim` that runs at exactly `fps` frames per second.
    matplotlib's own to_jshtml() rounds the frame interval down to a whole millisecond
    (1000 // fps = 41 ms at 24 fps, i.e. 3.94 s for 96 frames); here the interval is set to
    1000 / fps = 41.667 ms so that 96 frames last 4.00 s. If the player's code layout ever
    differs, the unmodified player is returned."""
    html = anim.to_jshtml(fps=fps)
    return re.sub(r"(new Animation\(frames, img_id, slider_id, )[0-9.]+(,)",
                  lambda m: "%s%.4f%s" % (m.group(1), 1000.0 / fps, m.group(2)), html, count=1)


def export_csv(path, grid, frames, L_cm, x_co2, x_n2o, flat=False, t_source=None, notes=""):
    """Long-format CSV: one row per (frame, wavelength). Comment lines start with '#'."""
    t_source = cfg.T_SOURCE_K if t_source is None else t_source
    I0, _unit = make_incident_spectrum(grid, flat, t_source)
    unit = ("arbitrary units (flat, I0 = 1)" if flat
            else "W m^-2 sr^-1 um^-1 (blackbody %.0f K)" % t_source)
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    cols = ["wavelength_um", "wavenumber_cm-1", "T_K", "P_Pa", "I0", "alpha_CO2_cm-1", "alpha_N2O_cm-1",
            "alpha_total_cm-1", "tau", "transmittance", "I_transmitted"]
    with open(path, "w") as fh:
        fh.write("# CO2+N2O transmission model; ILLUSTRATIVE simulation, not a validated product\n")
        fh.write("# x_CO2=%g x_N2O=%g L_cm=%g %s\n" % (x_co2, x_n2o, L_cm, notes))
        fh.write("# I0 unit: %s ; line parameters: HITRAN (see data/data_manifest.json); Voigt profile; Beer-Lambert\n"
                 % unit)
        fh.write(",".join(cols) + "\n")
        for fr in frames:
            tot = fr["a_co2"] + fr["a_n2o"]
            tau, trans, I = beer_lambert(tot, L_cm, I0)
            block = np.column_stack([1.0e4 / grid, grid, np.full_like(grid, fr["T"]), np.full_like(grid, fr["P"]),
                                     I0, fr["a_co2"], fr["a_n2o"], tot, tau, trans, I])
            np.savetxt(fh, block[::-1], delimiter=",", fmt="%.8g")      # ascending wavelength
    print("Wrote", path)

In [59]:
GRID = np.arange(cfg.NU_MIN, cfg.NU_MAX + 0.5 * cfg.NU_STEP, cfg.NU_STEP)
MODEL = {"obj": None, "dash": None, "key": None}        # the current computed SweepModel and its Dashboard


def draw(T, L_m=100.0, src="blackbody", lam_um=None, x_co2=None, x_n2o=None, view=None):
    """Static fallback (no widgets needed): one figure at temperature T [K], path length L_m [m]. Needs MODEL['obj']."""
    m = MODEL["obj"]
    d = Dashboard(GRID, m, src, view)
    d.render(T, L_m, m.x_co2_ref if x_co2 is None else x_co2, m.x_n2o_ref if x_n2o is None else x_n2o,
             m.lam_sel_um if lam_um is None else lam_um)
    display(d.fig)

if widgets is None:
    print("ipywidgets is not installed: the interactive controls are skipped. Use build_sweep_model(...) and draw(T, ...) instead.")
else:
    sty = {"description_width": "150px"}
    def wide(w=620):
        return widgets.Layout(width="%dpx" % w)          # one Layout PER widget (a shared one would hide several sliders)
    mode_dd = widgets.Dropdown(options=[("constant number density (P = n k T)", "isochoric"), ("constant pressure", "isobaric")],
                               value=cfg.SWEEP_MODE, description="Sweep", style=sty, layout=wide())
    n_sl = widgets.FloatLogSlider(value=cfg.N_DEFAULT_M3, base=10, min=cfg.N_BOUNDS_LOG10_M3[0], max=cfg.N_BOUNDS_LOG10_M3[1], step=0.01,
                                  description="Number density n (m^-3)", readout_format=".2e", continuous_update=False, style=sty, layout=wide())
    P_sl = widgets.FloatSlider(value=cfg.P_DEFAULT, min=cfg.P_BOUNDS[0], max=cfg.P_BOUNDS[1], step=cfg.P_STEP,
                               description="Pressure P (Pa)", readout_format=".0f", continuous_update=False, style=sty, layout=wide())
    T_rng = widgets.FloatRangeSlider(value=cfg.T_DEFAULT, min=cfg.T_BOUNDS[0], max=cfg.T_BOUNDS[1], step=1.0,
                                     description="T range of the sweep (K)", readout_format=".0f", continuous_update=False, style=sty, layout=wide())
    L_sl = widgets.FloatLogSlider(value=cfg.L_DEFAULT_M, base=10, min=cfg.L_BOUNDS_LOG10_M[0], max=cfg.L_BOUNDS_LOG10_M[1], step=0.05,
                                  description="Path length L (m)", readout_format=".3g", continuous_update=False, style=sty, layout=wide())
    xc_box = widgets.BoundedFloatText(value=round(cfg.X_CO2 * 1e6, 2), min=0.0, max=1e4, step=10.0, description="CO2 (ppm)", style=sty, layout=wide(300))
    xn_box = widgets.BoundedFloatText(value=round(cfg.X_N2O * 1e9, 2), min=0.0, max=1e6, step=10.0, description="N2O (ppb)", style=sty, layout=wide(300))
    src_dd = widgets.Dropdown(options=[("blackbody %.0f K" % cfg.T_SOURCE_K, "blackbody"), ("flat (I0 = 1)", "flat")], value="blackbody",
                              description="Incident I0", style=sty, layout=wide(300))
    lam_box = widgets.FloatText(value=round(1e4 / cfg.NU_MAX, 4), step=0.001, description="Selected wavelength (um)", style=sty, layout=wide(300))
    view_rng = widgets.FloatRangeSlider(value=(cfg.NU_MIN, cfg.NU_MAX), min=cfg.NU_MIN, max=cfg.NU_MAX, step=0.5, description="Zoom: wavenumber (cm^-1)",
                                        readout_format=".1f", continuous_update=False, style=sty, layout=wide())
    btn_compute = widgets.Button(description="Compute / update", button_style="primary", layout=wide(190))
    btn_movie = widgets.Button(description="Render movie (%d frames, %.0f s)" % (cfg.N_FRAMES, cfg.N_FRAMES / cfg.FPS), layout=wide(250))
    btn_csv = widgets.Button(description="Export CSV (current T)", layout=wide(210))
    T_slider = widgets.FloatSlider(value=cfg.T_DEFAULT[0], min=cfg.T_BOUNDS[0], max=cfg.T_BOUNDS[1], step=0.1, readout_format=".1f",
                                   description="Temperature (K)", continuous_update=True, style=sty, layout=wide(720))
    play = widgets.Play(min=0, max=cfg.N_PLAY - 1, step=1, interval=cfg.PLAY_INTERVAL_MS, value=0, description="Play")
    img = widgets.Image(format="png", layout=widgets.Layout(width="1000px"))
    out_status, out_movie = widgets.Output(), widgets.Output()
    BUSY = {"flag": False}

    def _set_range(sl, lo, hi):
        sl.min, sl.max = min(lo, sl.min), max(hi, sl.max)       # widen first, then narrow: never an invalid intermediate state
        sl.min, sl.max = lo, hi
        sl.value = min(max(sl.value, lo), hi)

    def get_dash():
        m = MODEL["obj"]
        key = (id(m), src_dd.value, view_rng.value)
        if MODEL["dash"] is None or MODEL["key"] != key:
            MODEL["dash"], MODEL["key"] = Dashboard(GRID, m, src_dd.value, view_rng.value), key
        return MODEL["dash"]

    def redraw(*_):
        if MODEL["obj"] is None or BUSY["flag"]:
            return
        t0 = time.time()
        d = get_dash()
        d.render(T_slider.value, L_sl.value, xc_box.value * 1e-6, xn_box.value * 1e-9, lam_box.value)
        img.value = d.png_bytes()                                # the image is replaced in place: no flicker, no cell clearing
        redraw.last_ms = 1000 * (time.time() - t0)
    redraw.last_ms = 0.0

    def on_mode(*_):
        iso = mode_dd.value == "isochoric"
        n_sl.layout.display = "" if iso else "none"
        P_sl.layout.display = "none" if iso else ""

    def on_compute(_=None):
        BUSY["flag"] = True
        try:
            with out_status:
                clear_output()
                x1, x2 = xc_box.value * 1e-6, xn_box.value * 1e-9
                for msg in dilution_warnings(x1, x2):
                    print(msg)
                lo, hi = T_rng.value
                print("Computing exact spectra on %d grid points (%s) ..." % (len(GRID), DATA_LABEL))
                MODEL["obj"] = build_sweep_model(GRID, CO2, N2O, mode_dd.value, n_sl.value, P_sl.value, lo, hi, x1, x2, PARTITION_FN, label=DATA_LABEL)
                m = MODEL["obj"]
                print("done: %d exact spectra, the rest interpolated (anchors %.0f K apart at most)." % (len(m.T_anchor), cfg.ANCHOR_DT_MAX))
                P_lo, P_hi = m.P(m.T_lo), m.P(m.T_hi)
                if min(P_lo, P_hi) < cfg.P_BOUNDS[0] or max(P_lo, P_hi) > cfg.P_BOUNDS[1]:
                    print("NOTE: pressure %.0f-%.0f Pa leaves the AFGL tropospheric range %.0f-%.0f Pa: a what-if experiment." % (P_lo, P_hi, *cfg.P_BOUNDS))
            MODEL["dash"] = None
            _set_range(T_slider, m.T_lo, m.T_hi)
            T_slider.value = 0.5 * (m.T_lo + m.T_hi)
            lam_box.value = round(m.lam_sel_um, 4)
        finally:
            BUSY["flag"] = False
        redraw()

    def on_play(change):
        m = MODEL["obj"]
        if m is not None:
            T_slider.value = m.T_lo + (m.T_hi - m.T_lo) * play.value / (cfg.N_PLAY - 1)

    def on_movie(_):
        if MODEL["obj"] is None:
            on_compute()
        with out_status:
            clear_output()
            print("Rendering %d frames at %d fps = %.1f s ... (about a minute)" % (cfg.N_FRAMES, cfg.FPS, cfg.N_FRAMES / cfg.FPS))
            anim = build_animation(GRID, MODEL["obj"], L_sl.value, xc_box.value * 1e-6, xn_box.value * 1e-9, src_dd.value, lam_box.value, view_rng.value)
            html = player_html(anim, cfg.FPS)
            print("done.")
        with out_movie:
            clear_output(wait=True)
            display(HTML(html))

    def on_csv(_):
        m = MODEL["obj"]
        if m is None:
            return
        fr = m.frame(T_slider.value, xc_box.value * 1e-6, xn_box.value * 1e-9)
        path = os.path.join("outputs", "spectrum_T%.1fK_P%.0fPa_L%.0fm.csv" % (fr["T"], fr["P"], L_sl.value))
        with out_status:
            export_csv(path, GRID, [fr], L_sl.value * 100.0, xc_box.value * 1e-6, xn_box.value * 1e-9, src_dd.value == "flat", cfg.T_SOURCE_K,
                       notes="exact=%s data=%s" % (fr["exact"], DATA_LABEL))

    mode_dd.observe(on_mode, names="value")
    on_mode()
    play.observe(on_play, names="value")
    for w in (T_slider, L_sl, src_dd, lam_box, xc_box, xn_box, view_rng):
        w.observe(redraw, names="value")
    btn_compute.on_click(on_compute)
    btn_movie.on_click(on_movie)
    btn_csv.on_click(on_csv)

    display(widgets.VBox([
        widgets.HTML("<b>Conditions</b> &nbsp;(sweep mode, density / pressure and temperature range need <i>Compute</i>; "
                     "temperature, path length, CO<sub>2</sub> / N<sub>2</sub>O amounts, incident spectrum, zoom and Play update instantly)"),
        mode_dd, n_sl, P_sl, T_rng, L_sl, widgets.HBox([xc_box, xn_box]), widgets.HBox([src_dd, lam_box]), view_rng,
        widgets.HBox([btn_compute, btn_movie, btn_csv]), out_status,
        widgets.HTML("<b>Live animation</b> &nbsp;(drag the temperature slider or press Play)"),
        widgets.HBox([play, T_slider]), img, widgets.HTML("<b>Pre-rendered movie</b>"), out_movie]))
    on_compute()                                     # first computation with the default settings

### Save the movie as a file (optional)

MP4 needs ffmpeg and lasts exactly 4.00 s (96 frames at 24 fps); a GIF needs Pillow and can only time frames in 10 ms steps. Set `SAVE_MOVIE = True` to run it.

In [41]:
SAVE_MOVIE = False          # set True to write co2_n2o_transmission.gif (Pillow) or .mp4 (ffmpeg)
if SAVE_MOVIE and MODEL["obj"] is not None:
    anim = build_animation(GRID, MODEL["obj"], L_sl.value, xc_box.value * 1e-6, xn_box.value * 1e-9, src_dd.value, lam_box.value, view_rng.value)
    anim.save("co2_n2o_transmission.gif", writer="pillow", fps=cfg.FPS)
    # anim.save("co2_n2o_transmission.mp4", writer="ffmpeg", fps=cfg.FPS, dpi=150)

## 9. Checks you should run on real data

**None of these has been run by the author** (no HITRAN data, HAPI or internet). `run_real_data_checks()` runs everything that is possible with your files and records the rest as **NOT RUN**:
real-data line-list integrity, coverage, band edges, band centres, step/wing/threshold convergence, interpolation vs direct calculation, physical limits, and the comparison with HAPI's Voigt routine (CO₂ and N₂O each use their own mole fraction; HAPI's own wing cut-off and weak-line handling differ from ours, so only line cores are compared).
`strongest_lines()` lists the strongest lines in the window.

In [43]:
# ===== Checks on REAL data (HITRAN files in data/, HAPI installed). Each one is recorded as PASS / FAIL / NOT RUN. =====
def strongest_lines(lines, n=8):
    sel = np.where((lines.nu >= cfg.NU_MIN) & (lines.nu <= cfg.NU_MAX))[0]
    top = sel[np.argsort(lines.S[sel])[::-1][:n]]
    print(lines.name, ": %d lines in the window" % len(sel))
    for k in top:
        print("  lambda = %.5f um (%.4f cm-1)   S = %.3e   E'' = %8.2f   gamma_air = %.4f   iso %d"
              % (1e4 / lines.nu[k], lines.nu[k], lines.S[k], lines.E_low[k], lines.gamma_air[k], lines.iso[k]))


def _strongest_window(lines, half=10.0):
    inw = (lines.nu >= cfg.NU_MIN + half) & (lines.nu <= cfg.NU_MAX - half)
    c = float(lines.nu[inw][np.argmax(lines.S[inw])])
    return (c - half, c + half)


def check_against_hapi(name="N2O", nu_a=None, nu_b=None, T_list=(296.0, 255.65), P_atm=0.533, x_self=None, verbose=True):
    """Compare our cross section with HAPI's absorptionCoefficient_Voigt for the same lines.
    Needs the HAPI-written table (data/<name>*.data + .header) and HAPI installed.
    x_self defaults to the mole fraction of THE GAS BEING TESTED (the old default used the N2O value for CO2 too).
    Equivalent settings: same T, p, diluent mixture {air: 1-x, self: x}, same grid step, pressure in atm. Not identical:
    HAPI uses its own default wing cut-off and no weak-line threshold, so we compare only the line CORES
    (points above 20 % of our maximum); our cut-off is continuous and 20 cm-1. Returns [(T, max_rel, median_rel)]."""
    import hapi as _h
    x_self = default_x(name) if x_self is None else x_self
    lines = CO2 if name == "CO2" else N2O
    if nu_a is None:
        nu_a, nu_b = _strongest_window(lines, 5.0)
    _h.db_begin(cfg.DATA_DIR)
    tables = [os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(cfg.DATA_DIR, name + "*.data"))
              if os.path.exists(os.path.splitext(p)[0] + ".header")]
    if not tables:
        raise FileNotFoundError("no HAPI table (.data + .header) for %s in %s" % (name, cfg.DATA_DIR))
    g = np.arange(nu_a, nu_b, cfg.NU_STEP)
    out = []
    for T in T_list:
        mine = species_cross_section(g, lines, T, P_atm * ATM_PA, x_self, hapi_partition_fn, wing_cm1=cfg.WING_CM1, s_min=0.0,
                                     subtract_edge=cfg.SUBTRACT_EDGE)
        nu_h, sig_h = _h.absorptionCoefficient_Voigt(SourceTables=[tables[0]], Environment={"T": T, "p": P_atm},
                       WavenumberRange=(nu_a, nu_b), WavenumberStep=cfg.NU_STEP, HITRAN_units=True,
                       Diluent={"air": 1.0 - x_self, "self": x_self})
        ref = np.interp(g, nu_h, sig_h)
        core = mine > 0.2 * mine.max()
        rel = np.abs(mine[core] - ref[core]) / ref[core]
        out.append((T, float(rel.max()), float(np.median(rel))))
        if verbose:
            print("%s T=%.2f K, x_self=%.3g: core points %d, max |rel diff| = %.3e, median = %.3e" % (name, T, x_self, core.sum(), rel.max(), np.median(rel)))
    return out


def run_real_data_checks(hapi_ok_threshold=1e-2):
    """Run every real-data check that is possible and record the rest as NOT RUN with the reason."""
    have_data = "CO2" in globals() and globals().get("DATA_LABEL") == "HITRAN line data"
    have_hapi = hapi is not None
    names = [(CAT_REAL, "line-list integrity (finite positive S, isotopologues known, duplicates reported)"),
             (CAT_REAL, "spectral coverage of window + wings (data_manifest.json)"),
             (CAT_REAL, "bands not cut by the window (strong lines near edges)"),
             (CAT_REAL, "band centres: N2O nu3 ~2224, CO2 nu3 ~2349 cm-1"),
             (CAT_REAL, "numerical convergence: step, wing, S_min"),
             (CAT_REAL, "interpolation vs direct calculation at mid-anchor T"),
             (CAT_REAL, "physical limits: opaque CO2 line core (10 km), transparent (0.1 mm)"),
             (CAT_HAPI, "CO2 cross section vs HAPI Voigt (296 K and 255.65 K)"),
             (CAT_HAPI, "N2O cross section vs HAPI Voigt (296 K and 255.65 K)")]
    if not have_data or not have_hapi:
        why = "no HITRAN files in '%s'" % cfg.DATA_DIR if not have_data else "HAPI not installed (partition sums needed)"
        for c, n in names:
            record(c, n, "NOT RUN", why)
        print("Real-data checks NOT RUN:", why)
        return

    def attempt(cat, name, fn):
        try:
            status, metric = fn()
        except Exception as exc:
            status, metric = "NOT RUN", "could not run: %r" % (exc,)
        record(cat, name, status, metric)
        print("%-8s %s | %s" % (status, name, metric))

    def integrity():
        bad, dup_total = [], 0
        for nm, M, L in (("CO2", 2, CO2), ("N2O", 4, N2O)):
            ok = np.all(np.isfinite(L.S)) and np.all(L.S > 0) and set(np.unique(L.iso)) <= set(L.mass_amu)
            if not ok:
                bad.append(nm)
            dup_total += load_linelist(nm, M, cfg.DATA_DIR)[1]["duplicates"]["exact_duplicates_removed"]
        return ("PASS" if not bad else "FAIL"), "bad lists: %s; exact duplicates removed while loading: %d" % (bad or "none", dup_total)

    def coverage():
        res = [check_coverage(L, cfg.NU_MIN, cfg.NU_MAX, cfg.WING_CM1, os.path.join(cfg.DATA_DIR, "data_manifest.json")) for L in (CO2, N2O)]
        if any(r["confirmed"] is False for r in res):
            return "FAIL", "download range does not cover window + wing"
        if any(r["confirmed"] is None for r in res):
            return "NOT RUN", "no data_manifest.json, coverage cannot be confirmed from the line list alone"
        return "PASS", "download %s covers needed %s" % (res[0]["downloaded"], res[0]["need"])

    def edges():
        r = [check_band_edges(L, cfg.NU_MIN, cfg.NU_MAX) for L in (CO2, N2O)]
        n = sum(x["strong_lines_near_edges"] for x in r)
        return ("PASS" if n == 0 else "FAIL"), "lines with S >= 1e-3 S_max within 5 cm-1 of an edge: CO2 %d, N2O %d" % (r[0]["strong_lines_near_edges"], r[1]["strong_lines_near_edges"])

    def centres():
        out, ok = [], True
        for L, c in ((N2O, 2224.0), (CO2, 2349.0)):
            sel = (L.nu >= cfg.NU_MIN) & (L.nu <= cfg.NU_MAX) & (L.S >= 1e-2 * L.S.max())
            cen = float(np.sum(L.nu[sel] * L.S[sel]) / np.sum(L.S[sel]))
            ok &= abs(cen - c) < 12.0
            out.append("%s %.1f" % (L.name, cen))
        return ("PASS" if ok else "FAIL"), "S-weighted centre of strong lines: " + ", ".join(out)

    def conv():
        worst_dT = worst_ew = worst_smin = 0.0
        for L in (N2O, CO2):
            res = convergence_study(CO2, N2O, PARTITION_FN, _strongest_window(L), [(296.0, 101325.0), (255.0, 54020.0), (190.0, 9000.0)],
                                    L_m_list=(1.0, 100.0), verbose=False)
            for (name, T, Lm), v in res.items():
                if name == "baseline":
                    worst_dT, worst_ew = max(worst_dT, v[0]), max(worst_ew, v[1])
                    worst_smin = max(worst_smin, abs(res[("S_min x10", T, Lm)][0] - v[0]))
        return ("PASS" if worst_dT < 1e-2 and worst_ew < 1e-2 else "FAIL"), \
            "baseline vs reference: max |dTransmittance| %.1e, max rel. dEW %.1e; S_min x10 changes |dT| by %.1e" % (worst_dT, worst_ew, worst_smin)

    def interp():
        r = interpolation_validation(CO2, N2O, PARTITION_FN, GRID, cfg.SWEEP_MODE, cfg.N_DEFAULT_M3, cfg.P_DEFAULT, cfg.T_DEFAULT[0], cfg.T_DEFAULT[1],
                                     cfg.X_CO2, cfg.X_N2O, dT_list=(cfg.ANCHOR_DT_MAX,), verbose=False)
        eT, ea = r[cfg.ANCHOR_DT_MAX]
        return ("PASS" if eT < 5e-4 else "FAIL"), "anchors every %.0f K, mid-points, L = 1 m, 100 m, 10 km: max |dTransmittance| %.1e, max rel. d alpha %.1e" % (cfg.ANCHOR_DT_MAX, eT, ea)

    def limits():
        c = float(CO2.nu[(CO2.nu >= cfg.NU_MIN) & (CO2.nu <= cfg.NU_MAX)][np.argmax(CO2.S[(CO2.nu >= cfg.NU_MIN) & (CO2.nu <= cfg.NU_MAX)])])
        g = np.arange(c - 0.2, c + 0.2, 0.0005)
        a1, a2 = absorption_coefficients(g, CO2, N2O, 296.0, 101325.0, cfg.X_CO2, cfg.X_N2O, PARTITION_FN, wing_cm1=cfg.WING_CM1,
                                         s_min=cfg.S_MIN, subtract_edge=cfg.SUBTRACT_EDGE)
        t_thick, t_thin = np.exp(-(a1 + a2) * 1e6).min(), np.exp(-(a1 + a2) * 1e-2).min()
        return ("PASS" if t_thick < 1e-6 and t_thin > 0.99 else "FAIL"), "min transmittance at the strongest CO2 line: %.1e (10 km), %.4f (0.1 mm)" % (t_thick, t_thin)

    def hapi_check(name):
        def f():
            res = check_against_hapi(name, verbose=False)
            worst, med = max(r[1] for r in res), max(r[2] for r in res)
            return ("PASS" if med < hapi_ok_threshold else "FAIL"), "cores: max |rel diff| %.2e, median %.2e (PASS if median < %.0f %%)" % (worst, med, 100 * hapi_ok_threshold)
        return f

    attempt(*names[0], integrity); attempt(*names[1], coverage); attempt(*names[2], edges); attempt(*names[3], centres)
    attempt(*names[4], conv); attempt(*names[5], interp); attempt(*names[6], limits)
    attempt(*names[7], hapi_check("CO2")); attempt(*names[8], hapi_check("N2O"))

In [44]:
strongest_lines(CO2); strongest_lines(N2O)
run_real_data_checks()      # runs what is possible with your files; everything else is recorded as NOT RUN

CO2 : 74316 lines in the window
  lambda = 4.23466 um (2361.4659 cm-1)   S = 3.542e-18   E'' =   106.13   gamma_air = 0.0742   iso 1
  lambda = 4.23709 um (2360.1090 cm-1)   S = 3.515e-18   E'' =    81.94   gamma_air = 0.0755   iso 1
  lambda = 4.23227 um (2362.7980 cm-1)   S = 3.466e-18   E'' =   133.44   gamma_air = 0.0731   iso 1
  lambda = 4.23957 um (2358.7272 cm-1)   S = 3.375e-18   E'' =    60.87   gamma_air = 0.0770   iso 1
  lambda = 4.28097 um (2335.9194 cm-1)   S = 3.330e-18   E'' =   106.13   gamma_air = 0.0742   iso 1
  lambda = 4.22993 um (2364.1053 cm-1)   S = 3.304e-18   E'' =   163.87   gamma_air = 0.0722   iso 1
  lambda = 4.28420 um (2334.1563 cm-1)   S = 3.281e-18   E'' =   133.44   gamma_air = 0.0731   iso 1
  lambda = 4.27779 um (2337.6580 cm-1)   S = 3.278e-18   E'' =    81.94   gamma_air = 0.0755   iso 1
N2O : 46454 lines in the window
  lambda = 4.47183 um (2236.2234 cm-1)   S = 1.002e-18   E'' =   100.55   gamma_air = 0.0777   iso 1
  lambda = 4.47328 um (2235

## 10. Export

The **Export CSV** button writes the current frame. To write *all* frames of the movie in one long CSV (wavelength ascending), run the commented line below.

In [46]:
# export_csv("outputs/all_frames.csv", GRID, [MODEL["obj"].frame(T, xc_box.value*1e-6, xn_box.value*1e-9) for T in MODEL["obj"].temperatures(cfg.N_FRAMES)], L_sl.value * 100.0, xc_box.value*1e-6, xn_box.value*1e-9, src_dd.value == "flat", cfg.T_SOURCE_K)

## 11. References (what each source supports)

* Anderson, Clough, Kneizys, Chetwynd, Shettle, *AFGL Atmospheric Constituent Profiles (0–120 km)*, AFGL-TR-86-0110, Environmental Research Papers No. 954 (1986), https://apps.dtic.mil/sti/pdfs/ADA175173.pdf —
  the six reference atmospheres used for the layer choice and the temperature/pressure ranges; CO₂ and N₂O mixing-ratio profiles (the 1986 mixing ratios themselves are out of date, current values are from NOAA below).
* *U.S. Standard Atmosphere, 1976*, U.S. Government Printing Office, Washington, DC — $T_0=288.15$ K, $P_0=101325$ Pa, 6.5 K/km lapse rate, tropopause 11 km (216.65 K, 22 632 Pa).
* NOAA GML, CO₂ trends (423.03 ppm, Sep 2025): https://gml.noaa.gov/ccgg/trends/global.html · N₂O trends (339.76 ppb, Mar 2026): https://www.gml.noaa.gov/ccgg/trends_n2o
* HITRAN, *Definitions and units*, https://hitran.org/docs/definitions-and-units/ — units of $S$, $\gamma$ (HWHM at 296 K and 1 atm), $n_{air}$, and the temperature scaling of $S$.
* Kochanov et al., "HITRAN Application Programming Interface (HAPI)…", JQSRT **177**, 15–30 (2016), doi:10.1016/j.jqsrt.2016.03.005; https://hitran.org/hapi/ — HAPI; key requirement and daily request limit.
* Gordon et al., "The HITRAN2024 molecular spectroscopic database", JQSRT **353**, 109807 (2026), doi:10.1016/j.jqsrt.2026.109807 — database edition (as listed on the HAPI page).
* Gamache et al., "Partition sums for molecules and their isotopologues for HITRAN2024", JQSRT **345**, 109568 (2025), doi:10.1016/j.jqsrt.2025.109568 — TIPS partition sums used by HAPI.
* Hargreaves et al., "Spectroscopic line parameters of NO, NO₂, and N₂O for the HITEMP database", JQSRT **232**, 35–53 (2019) — high-temperature N₂O (¹⁴N₂¹⁶O only).
* Hashemi et al., HITRAN2020 air/self-broadening updates for N₂O and CO, and "Revising the line-shape parameters for air- and self-broadened CO₂ lines toward a sub-percent accuracy level" (2020) — origin of the broadening parameters (journal details not verified here).
* Doppler width, the Voigt profile via the Faddeeva function (SciPy `wofz`), Beer–Lambert law, and the Planck function are standard results used without a verified equation number. **No equation or page numbers are claimed.**
* HAPI manual, https://hitran.org/static/hapi/hapi_manual.pdf — states that the default `.data` format is the 160-character HITRAN format (basis of the file reader); `absorptionCoefficient_Voigt`, `Diluent={'air':…,'self':…}`, `partitionSum(M, I, T)` are HAPI functions as listed in its documentation. Keyword details for wing cut-off were **not** verified and are not relied on.
* U.S. Standard Atmosphere 1976: geopotential vs geometric height, r₀ = 6356.766 km (used in `geopotential_from_geometric_m`; verified numerically against the AFGL U.S. Standard rows).
* Band centres (N₂O ν₃ ≈ 2224 cm⁻¹, CO₂ ν₃ ≈ 2349 cm⁻¹) and the rotational constants used for the window estimate are standard spectroscopic values quoted without a verified page reference; `check_band_edges()` and `window_content()` test them against your line list. The Ladenburg–Reiche square-root limit $W=2\sqrt{S N\gamma_L}$ is a standard result, checked here numerically.
* No equation or page numbers are claimed.

## 12. Remaining limitations and unverified features

* **Not run by the author:** all real-HITRAN checks, all HAPI comparisons, the HAPI download, the widgets in a live Jupyter session, the movie/MP4 export. Real-data numbers (line counts, band-edge result, convergence errors on real lines, interpolation error on real lines) are unknown until you run §9.
* The wing cut-off (20 cm⁻¹) is a **modelling choice**: real CO₂ far wings are sub-Lorentzian and affected by line mixing and collision-induced absorption, none of which is modelled. The convergence study measures sensitivity to the cut-off, not agreement with experiment.
* The AFGL values for the tropical, mid-latitude winter and subarctic winter rows, and the NOAA mole fractions, are carried over from the previous notebook and were not re-verified.
* Uniform slab only (one T, one P). Vertical paths, water vapour, clouds, scattering, emission, instrument line shape are not modelled. Sweeps are what-if experiments, not atmospheric profiles.
* Air broadening is used for all non-self collisions (no CO₂/N₂O cross-broadening in HITRAN); pressure shift uses δ_air only (no temperature dependence of the shift); HITRAN lacks weak hot-band lines (HITEMP for hot gas).
* Pressure/density, sweep mode and T range need a recomputation; changing the gas amounts rescales α linearly (small, measured approximation).
* The `.data` reader relies on HAPI's documented default 160-character format; a HAPI table written with custom parameter groups is not supported.

In [48]:
# Validation table: synthetic software tests, numerical convergence (synthetic), real HITRAN data, HAPI comparison
validation_table()

category                                 check                                                status   metric
------------------------------------------------------------------------------------------------------------------------------------------------------
HAPI comparison                          CO2 cross section vs HAPI Voigt (296 K and 255.65 K) PASS     cores: max |rel diff| 4.40e-03, median 1.76e-03 (PASS if median < 1 %)
HAPI comparison                          N2O cross section vs HAPI Voigt (296 K and 255.65 K) PASS     cores: max |rel diff| 9.31e-03, median 3.65e-03 (PASS if median < 1 %)
Numerical convergence (synthetic lines)  conv_grid_wing_and_threshold_study_on_synthetic_lines PASS     baseline (step 0.0025, wing 20 cm-1) vs reference: max |dTransmittance| 4.7e-03, max rel. error of equivalent width 8.8e-03; going from step 4x coarser to baseline changes |dTransmittance| by < 3e-06
Numerical convergence (synthetic lines)  conv_interpolation_error_falls_as_anchor_spaci